# OCR nguồn ảnh TV02 bằng PaddleOCR + VietOCR — Kaggle / Google Colab

**Kaggle:** Import notebook → Settings → Accelerator → GPU và bật Internet.
**Colab:** File → Upload notebook → Runtime → Change runtime type → GPU.
Kaggle đọc Drive qua OAuth trong Kaggle Secrets; xem phần 4 và helper `tao_quyen_drive.py`.
Notebook đọc đúng thư mục Drive bằng **ID**, không cần đặt shortcut hoặc tải lại dataset.
Chỉ tạo OCR thô: giữ dự đoán, điểm tin cậy và tọa độ; chưa sửa chữ, bỏ chú thích, tách câu hay dóng hàng.
Model được tải khi bạn chạy cell nạp model. Notebook này không clone repository.

**Thứ tự chạy:** cấu hình → cài thư viện → kết nối Drive → kiểm kê → nạp model → thử một ảnh → chạy toàn bộ → ghép theo tác phẩm → xem báo cáo.

### Phạm vi TV02 đã đối chiếu với `phancong.md` và Drive ngày 12/11/2026

| Nguồn | Phần TV02 | Phiên bản |
|---|---|---|
| Văn sách thi Đình, CPO_11 | Bài 14–18, PDF 188–365 | Hán, dịch âm, dịch nghĩa |
| Thơ văn Lý–Trần, CPO_12 | Bài 131–207, PDF 187–366 | Hán, dịch âm, dịch nghĩa, dịch thơ |
| Bắc sứ, CPO_09 | Hán PDF 49–101 | 43 ảnh sau khi bỏ 10 trang đã được yêu cầu |
| Bắc sứ Việt, CPO_09 | PDF 32–61, đúng mốc đầu/cuối phân công | 2 TXT nguồn thô: **sao chép nguyên byte, không OCR** |

Hịch tướng sĩ là phần mẫu chung do TV1 chuẩn bị; không tự thêm vào việc OCR riêng của TV02.
Tên quyển 1/quyển 4 của Bắc sứ giữ theo thư mục đã có, không thay đổi phạm vi trang.
Hai phạm vi Hán và Việt không phải ánh xạ dịch theo số trang. OCR ở bước 4 tách biệt với nguồn bước 3.

Nguồn Drive: [thư mục nhóm](https://drive.google.com/drive/folders/1WFpIQDOkccBRP94FY175P-kiY0QtysnZ).
Có 10 cặp thư mục dịch nghĩa Lý–Trần trùng tên, một bên rỗng khi kiểm tra. Không xóa hoặc sửa Drive nguồn.
**Hai nhánh:** Hán dùng PaddleOCR; ảnh Việt dùng Paddle detection + VietOCR.
Các dòng Việt được cắt trong bộ nhớ. TXT nguồn Bắc sứ Việt không OCR.


### CLC LAB OCR
Run the next cell by itself for Thi Dinh 24 Han v03. It reads `CLC_OCR_API_KEY` from `.env.clc_ocr` and writes TXT/JSON under `san_pham/TV04`.

In [ ]:
# CLC LAB OCR - Thi Dinh bai 24, Han v03
from pathlib import Path
import sys

here = Path.cwd().resolve()
project = next(p for p in (here, *here.parents) if (p / "ten_file.py").is_file())
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
from cong_cu.clc_ocr import run

print(run(project))


## 1. Cấu hình chung — sửa tại đây

- `INPUT_MODE`: `drive_api` đọc thư mục chia sẻ bằng ID; `local` đọc thư mục đã tải/mount.
- `OUTPUT_ROOT`: nơi lưu kết quả (Kaggle working hoặc Colab Drive); `RUN_NAME`: phiên bản kết quả. Khi đổi model/tham số, dùng tên phiên bản mới.
- `ONLY_WORKS`: để `{}` lấy tất cả bài thuộc phạm vi; ví dụ `{"CPO_11": ["bai0014"], "CPO_12": ["bai0131"]}`.
- `MAX_IMAGES_PER_JOB`: đặt `3` để thử một nhóm nhỏ; `None` chạy toàn bộ. TXT có sẵn vẫn được sao chép.
- `HAN_LAYOUTS`: chiều đọc riêng từng nguồn Hán; `HAN_IMAGE_OVERRIDES` dành cho ảnh ngoại lệ.

Ảnh nguồn không bị cắt/chuẩn hóa lại. Giữ ảnh scan chất lượng gốc, không dùng ảnh xem trước DPI thấp.
Nhánh Việt chỉnh `VIETOCR_MODEL`, `VIETOCR_WEIGHTS`, `VIETOCR_MAX_WIDTH`, `VIETOCR_MAX_LENGTH`,
`VIET_CROP_PADDING`, `VIET_READING_ORDER`, `VIET_REVIEW_CONFIDENCE`.
Tham số box/unclip áp dụng cho detector Việt; `REC_SCORE_THRESHOLD` áp dụng cho Hán cột dọc.
Dùng RUN_NAME mới khi đổi model/crop/thứ tự để giữ kết quả trước đó.


Chiều đọc đã rà theo nguồn: `thi_dinh_han=vertical_rtl`, `ly_tran_han=horizontal_ltr`,
`bac_su_han=vertical_rtl`. `VIET_READING_ORDER` vẫn áp dụng riêng nhánh Việt.
`RUN_NAME` nền `paddle_v6b1_layout_v03` tự thêm mode Bắc sứ để tách đầu ra phiên mới.
Các ngoại lệ hình học trong `HAN_IMAGE_OVERRIDES` chỉ áp dụng đúng file đã chỉ định;
đổi dữ liệu thì kiểm tra lại preview/ngoại lệ. Tách hàng là heuristic, không xác nhận đã đọc đúng chữ.


Bắc sứ Hán trong luồng chính dùng `BAC_PREPROCESS_MODE="gray_scale"` hoặc `"binary"`.
Tham số chính nằm trong `BAC_MAIN_PREPROCESS_PARAMS`, độc lập tham số thử ở 10e.
RUN_NAME tự thêm mode; Bắc sứ Việt TXT và các sách khác giữ luồng hiện có.


In [ ]:
from pathlib import Path
import importlib.util
import os
import hashlib
import io
import json
import re
import shutil
import subprocess
import sys
import time
import uuid
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version, distributions

def detect_runtime():
    if Path("/kaggle/working").is_dir() or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "colab"
    except (ImportError, ValueError):
        pass
    return "local"

RUNTIME = detect_runtime()               # Có thể đặt rõ "kaggle" hoặc "colab".
INPUT_MODE = "local"                 # Hoặc "local".
here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / "ten_file.py").is_file()), None)
LOCAL_INPUT_ROOT = (Path("/kaggle/input") if RUNTIME == "kaggle" else
                    Path("/content/anh_ocr") if RUNTIME == "colab" else
                    project / "data" if project else here / "data")
OUTPUT_ROOT = (Path("/kaggle/working/NLP_OCR_TV04") if RUNTIME == "kaggle"
               else Path("/content/drive/MyDrive/NLP_OCR_TV04") if RUNTIME == "colab"
               else Path.cwd() / "ocr_output")
LOCAL_CACHE = Path("/kaggle/temp/nlp_ocr_cache" if RUNTIME == "kaggle" else "/content/nlp_ocr_cache")
DRIVE_AUTH_MODE = "oauth"                # Kaggle: "oauth" hoặc "service_account".
DRIVE_SECRET_NAME = "GOOGLE_DRIVE_CREDENTIALS"  # Tên Kaggle Secret, không phải giá trị bí mật.
RUN_NAME = "paddle_v6b1_layout_v03"
DEVICE = "gpu:0"                        # Đổi thành "cpu" nếu không có GPU.
INFERENCE_ENGINE = "transformers"       # PP-OCRv6 qua PyTorch có sẵn.
TRANSFORMERS_VERSION = "5.23.0"
PADDLEOCR_VERSION = "3.7.0"
PADDLEX_VERSION = "3.7.2"
DETECTION_MODEL = "PP-OCRv6_medium_det"
RECOGNITION_MODEL = "PP-OCRv6_medium_rec"   # Chỉ nhánh Hán.
HAN_REC_BATCH_SIZE = 1                    # Giữ kết quả batch 1 đã kiểm tra.
VIETOCR_VERSION = "0.3.13"
VIETOCR_MODEL = "vgg_transformer"           # Hoặc "vgg_seq2seq".
VIETOCR_CONFIG_REVISION = "fe8c3a7fc714aec57ab81cec844eb3adf0c1636c"
VIETOCR_WEIGHTS = ""                        # Rỗng: checkpoint chính thức; hoặc đường dẫn .pth.
VIETOCR_MAX_WIDTH = 512                     # Chiều rộng tối đa, cao 32; giữ tỉ lệ dòng ngắn.
VIETOCR_MAX_LENGTH = 256                    # Giới hạn giải mã; dòng quá dài được đánh dấu.
VIET_READING_ORDER = "top_to_bottom"         # Một cột: trên→dưới; "model": giữ thứ tự detector.
VIET_CROP_PADDING = 2                       # Nới vùng dòng theo pixel ảnh nguồn.
VIET_REVIEW_CONFIDENCE = 0.80               # Ngưỡng riêng, không so trực tiếp với Paddle.
DETECTION_BOX_THRESHOLD = 0.60
DETECTION_UNCLIP_RATIO = 1.50

ONLY_WORKS = {}                         # {} hoặc {"CPO_11": ["bai0014"]}.
MAX_IMAGES_PER_JOB = None               # Ví dụ 3 để chạy thử; None để chạy hết.
# Chiều đọc Hán của từng nguồn; không dùng một thứ tự chung cho mọi sách.
HAN_LAYOUTS = {
    "thi_dinh_han": "vertical_rtl",     # Trong cột: trên→dưới; các cột: phải→trái.
    "ly_tran_han": "horizontal_ltr",    # Trong hàng: trái→phải; các hàng: trên→dưới.
    "bac_su_han": "vertical_rtl",
}
# Chỉ dùng cho horizontal_ltr. Lọc nhiễu trên ảnh phân tích, crop nhận dạng giữ pixel nguồn.
HAN_ROW_SETTINGS = dict(max_side=2048, ink_threshold=160, min_ink_fraction=0.02,
    bridge_gap=2, min_height_fraction=0.008, padding_fraction=0.25, min_padding_fraction=0.02)
# Ngoại lệ theo đúng job/relative_path, không theo ID detection dùng chung cho mọi ảnh.
# Ví dụ khóa: "ly_tran_han/CPO_12_TV02_bai0131_han_v01/phan0001_pdfp0187.png"
# Giá trị: {"row_settings": {"min_ink_fraction": 0.015}}
# Hoặc {"row_boxes": [[x1, y1, x2, y2], ...]} dùng tọa độ pixel trên ảnh nguồn.
# Có thể ghi "layout" hoặc "order_indices" (hoán vị đầy đủ ID vùng) cho trang ngoại lệ.
# Ngoại lệ hình học đã xem trên ảnh TV02: trang nhiễu và đề mục ngắn.
# Không có chữ dự đoán/bản chuẩn trong các quy tắc này.
HAN_IMAGE_OVERRIDES = {}
REC_SCORE_THRESHOLD = 0.0               # Không chủ động lọc bỏ chữ điểm thấp.
REVIEW_CONFIDENCE = 0.80                # Chỉ đánh dấu cần xem lại, không xóa chữ.
DETECTION_MAX_SIDE = 2048               # Tăng nếu chữ nhỏ bị sót; tốn thêm bộ nhớ.
SAVE_JSON = True                        # JSON cần cho truy vết và kiểm tra thứ tự.

assert INPUT_MODE in {"drive_api", "local"}
assert set(HAN_LAYOUTS.values()) <= {"vertical_rtl", "horizontal_ltr"}
assert isinstance(HAN_REC_BATCH_SIZE, int) and HAN_REC_BATCH_SIZE > 0
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME), "RUN_NAME chỉ dùng chữ, số, _ hoặc -."
assert MAX_IMAGES_PER_JOB is None or MAX_IMAGES_PER_JOB > 0
assert VIETOCR_MODEL in {"vgg_transformer", "vgg_seq2seq"}
assert VIET_READING_ORDER in {"top_to_bottom", "model"}
assert VIETOCR_MAX_WIDTH >= 32 and VIETOCR_MAX_LENGTH > 0 and VIET_CROP_PADDING >= 0
assert SAVE_JSON, "Giữ JSON để truy vết tọa độ và điểm tin cậy."
# Bắc sứ Hán trong luồng chính: chỉ hai mode, không áp dụng sang sách khác/bản Việt.
BAC_PREPROCESS_MODE = "gray_scale"       # "gray_scale" hoặc "binary".
BAC_COLUMN_CENTER_RATIO = 0.45          # Dung sai gom cột theo bề rộng vùng dọc điển hình.
BAC_MAIN_PREPROCESS_PARAMS = {'red_hue_width': 18,
 'red_min_saturation': 50,
 'red_min_r_minus_g': 20,
 'red_protect_max_below': 100,
 'background_max_side': 768,
 'background_filter_size': 31,
 'background_blur': 5.0,
 'black_point': 65,
 'white_point': 195,
 'tone_gamma': 1.15,
 'binary_threshold': None}
assert BAC_PREPROCESS_MODE in {"gray_scale", "binary"}
assert 0 < BAC_COLUMN_CENTER_RATIO < 1
# Tự tách thư mục khi đổi mode. Đổi thêm RUN_NAME nếu chỉnh tham số trong cùng mode.
RUN_NAME += "_bac_" + BAC_PREPROCESS_MODE

RUN_ROOT = OUTPUT_ROOT / RUN_NAME

## 2. Chọn tác phẩm, phiên bản và thư mục dữ liệu

`SELECTED_JOBS = list(JOBS)` chọn tất cả phần TV02. Muốn chạy riêng, đổi thành ví dụ
`SELECTED_JOBS = ["bac_su_han"]` hoặc `["thi_dinh_han", "thi_dinh_dich_nghia"]`.

Muốn đổi bộ ảnh: sửa `folder_id` hoặc `local_folder`. Khi đổi sang phần việc khác, chỉnh cả
`member`, `work_range`, `page_range`, `exclude_pages`; các trường này bảo vệ khỏi lấy nhầm bài.
Muốn thêm tác phẩm, sao chép một cấu hình và đặt `key` mới. `mode="text"` luôn giữ TXT nguyên byte.

In [ ]:
JOBS = {
    "thi_dinh_han": dict(code="CPO_11", kind="han", mode="ocr", member=4,
        folder_id="1hps7wU_3W9zKuMIS8sODOELqIvl9q9Nj", local_folder="ThiDinh/01_nguon/han",
        work_range=(23, 24), page_range=(565, 698), exclude_pages=[]),
    "thi_dinh_dich_am": dict(code="CPO_11", kind="dich_am", mode="ocr", member=4,
        folder_id="1fsilAL2E6RxMbGwbr1Os5u1VuU8NbkGo", local_folder="ThiDinh/01_nguon/viet/dich_am",
        work_range=(23, 24), page_range=(565, 698), exclude_pages=[]),
    "thi_dinh_dich_nghia": dict(code="CPO_11", kind="dich_nghia", mode="ocr", member=4,
        folder_id="1ECML6kWa1HNNrzygZxV-r5jmh7HEYFMA", local_folder="ThiDinh/01_nguon/viet/dich_nghia",
        work_range=(23, 24), page_range=(565, 698), exclude_pages=[]),
    "ly_tran_han": dict(code="CPO_12", kind="han", mode="ocr", member=4,
        folder_id="1gO2-KPt2WYeJlht5o2-4K_oASiSrtmY5", local_folder="LyTran/01_nguon/han",
        work_range=(314, 415), page_range=(526, 749), exclude_pages=[]),
    "ly_tran_dich_am": dict(code="CPO_12", kind="dich_am", mode="ocr", member=4,
        folder_id="1pThBo58orAvzNMNIhkt_uxkVUTSBfXWv", local_folder="LyTran/01_nguon/viet/dich_am",
        work_range=(314, 415), page_range=(526, 749), exclude_pages=[]),
    "ly_tran_dich_nghia": dict(code="CPO_12", kind="dich_nghia", mode="ocr", member=4,
        folder_id="1_NUL5agMtARjNEXnUdRMrxofqJNo-YYj", local_folder="LyTran/01_nguon/viet/dich_nghia",
        work_range=(314, 415), page_range=(526, 749), exclude_pages=[]),
    "ly_tran_dich_tho": dict(code="CPO_12", kind="dich_tho", mode="ocr", member=4,
        folder_id="1a40Oq7480dtI605NT6m-ylcnvT0Vus07", local_folder="LyTran/01_nguon/viet/dich_tho",
        work_range=(314, 415), page_range=(526, 749), exclude_pages=[]),
    "bac_su_han": dict(code="CPO_09", kind="han", mode="ocr", member=4,
        folder_id="1ceUCwOV3WvqwuAtSpsiEstteYLTBd6xO", local_folder="BacSu/01_nguon/han",
        work_range=None, page_range=(155, 207),
        exclude_pages=[]),
    "bac_su_viet_txt": dict(code="CPO_09", kind="dich_nghia", mode="text", member=4,
        folder_id="1PQZHzHXyUfsvpTyYfe3rhSeZpwjw-wyy", local_folder="BacSu/01_nguon/viet/dich_nghia",
        work_range=None, page_range=None, exclude_pages=[]),
}

SELECTED_JOBS = ["thi_dinh_han", "thi_dinh_dich_am", "thi_dinh_dich_nghia", "ly_tran_han", "ly_tran_dich_am", "ly_tran_dich_nghia", "ly_tran_dich_tho"]               # Hoặc ["bac_su_han"] để chạy một nguồn.
assert SELECTED_JOBS and len(set(SELECTED_JOBS)) == len(SELECTED_JOBS)
assert set(SELECTED_JOBS) <= set(JOBS)
for key in SELECTED_JOBS:
    print(key, "→", JOBS[key]["code"], JOBS[key]["kind"], JOBS[key]["mode"])

## 3. Kiểm tra PyTorch và cài SDK

PP-OCRv6 chạy qua `engine="transformers"`, dùng PyTorch có sẵn của Kaggle/Colab.
Không cài `paddlepaddle-gpu` hoặc thay Torch/CUDA/NCCL. Cell thử phép tính GPU trong tiến trình mới,
khóa phiên bản bộ GPU và kiểm tra kế hoạch pip trước khi cài SDK.

**Nếu đã chạy bộ cài Paddle GPU cũ:** lưu kết quả, Stop Session rồi tạo phiên Kaggle mới có GPU/Internet.
Restart kernel đơn thuần không phục hồi các gói đã bị hạ phiên bản. Dùng lại Secret Drive hiện có.
Chạy các cell từ đầu theo thứ tự; không chạy thẳng cell 8 sau khi mất kernel.
Không chạy cell cài đặt trên môi trường cá nhân đang dùng chung với dự án khác.

Ghim `transformers==5.23.0`: đã kiểm tra registry detection `pp_ocrv6_medium_det`
và recognition `pp_ocrv6_small_rec` (checkpoint **medium_rec** dùng model_type này).
Bản 5.8.1 thiếu kiến trúc v6; chỉ import được AutoModel chưa chứng minh model tương thích.
Sau khi đổi phiên bản Transformers trong phiên đang chạy, restart kernel rồi chạy từ đầu.

VietOCR 0.3.13 được cài `--no-deps` cùng các phụ thuộc inference riêng để tránh hạ Pillow
và kéo bộ augmentation/training cũ. Notebook chỉ phục vụ inference; `pip check` có thể báo các pin
training không khớp. Không dùng `pip install -U vietocr` để tự sửa các thông báo đó.


In [ ]:
DRIVE_READ_SCOPE = "https://www.googleapis.com/auth/drive.readonly"

def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


def is_gpu_stack_package(name):
    name = re.sub(r"[-_.]+", "-", name).lower()
    return (name in {"torch", "torchvision", "torchaudio", "triton"}
            or name.startswith(("nvidia-", "cuda-", "paddlepaddle")))


def gpu_stack_versions():
    return {d.metadata["Name"]: d.version for d in distributions()
            if d.metadata.get("Name") and is_gpu_stack_package(d.metadata["Name"])}


def probe_torch(device):
    if device != "cpu" and not re.fullmatch(r"gpu(?::\d+)?", device):
        raise ValueError("DEVICE phải là cpu, gpu hoặc gpu:0.")
    target = "cuda:" + device.split(":")[-1] if ":" in device else ("cuda:0" if device == "gpu" else "cpu")
    # Tiến trình mới tránh dùng nhầm thư viện GPU đã nạp trong kernel.
    code = """
import json, sys, torch
target = sys.argv[1]
if target.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('GPU chưa khả dụng; bật Settings > Accelerator > GPU.')
x = torch.ones(2, device=target)
assert (x + x).sum().item() == 4
print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'device': target}))
"""
    try:
        result = subprocess.run([sys.executable, "-c", code, target],
                                capture_output=True, text=True, check=False, timeout=60)
    except (OSError, subprocess.SubprocessError) as exc:
        raise RuntimeError("Không kiểm tra được PyTorch. Khởi tạo phiên Kaggle mới, bật GPU/Internet.") from exc
    if result.returncode != 0:
        detail = (result.stderr or result.stdout)[-2500:]
        raise RuntimeError("PyTorch/GPU của phiên chưa dùng được. Nếu đã cài Paddle GPU bằng cell cũ, "
                           "hãy lưu kết quả rồi Stop Session và tạo phiên Kaggle mới; restart kernel chưa khôi phục gói. "
                           "Bật Accelerator > GPU, hoặc chọn DEVICE='cpu' khi chủ động dùng CPU.\n" + detail)
    return json.loads(result.stdout.strip().splitlines()[-1])


def reject_gpu_changes(report):
    changes = [p["metadata"]["name"] for p in report.get("install", [])
               if is_gpu_stack_package(p["metadata"]["name"])]
    if changes:
        raise RuntimeError("Dừng cài đặt vì pip định thay đổi bộ GPU: " + ", ".join(changes))


def validate_drive_credentials(info, mode):
    if not isinstance(info, dict):
        raise ValueError("Kaggle Secret phải chứa nội dung JSON xác thực Google Drive.")
    if mode == "oauth":
        required = {"refresh_token", "client_id", "client_secret", "token_uri"}
        if not all(info.get(field) for field in required):
            raise ValueError("OAuth cần token có refresh_token; dùng tao_quyen_drive.py rồi cập nhật Kaggle Secret.")
        if info["token_uri"] not in {"https://oauth2.googleapis.com/token", "https://accounts.google.com/o/oauth2/token"}:
            raise ValueError("Địa chỉ làm mới quyền phải là endpoint OAuth của Google.")
        scopes = info.get("scopes")
        if isinstance(scopes, str):
            scopes = scopes.split()
        if scopes and DRIVE_READ_SCOPE not in scopes:
            raise ValueError("Quyền đã cấp chưa có drive.readonly; đăng nhập lại bằng helper.")
    elif mode == "service_account":
        if info.get("type") != "service_account" or not info.get("client_email") or not info.get("private_key"):
            raise ValueError("Secret chưa phải JSON service account hợp lệ.")
        if info.get("token_uri") != "https://oauth2.googleapis.com/token":
            raise ValueError("Service account phải dùng token endpoint của Google.")
    else:
        raise ValueError("DRIVE_AUTH_MODE phải là oauth hoặc service_account.")
    return info
# Chỉ tạo config và tra registry, không tải trọng số hoặc khởi tạo model.
TRANSFORMERS_COMPATIBILITY_CODE = r"""
import json
import transformers
from transformers import AutoConfig
from transformers.models.auto.modeling_auto import (
    MODEL_FOR_OBJECT_DETECTION_MAPPING_NAMES,
    MODEL_FOR_TEXT_RECOGNITION_MAPPING_NAMES,
)
required = [
    ("pp_ocrv6_medium_det", MODEL_FOR_OBJECT_DETECTION_MAPPING_NAMES),
    ("pp_ocrv6_small_rec", MODEL_FOR_TEXT_RECOGNITION_MAPPING_NAMES),
]
for model_type, mapping in required:
    config = AutoConfig.for_model(model_type)
    if model_type not in mapping:
        raise RuntimeError("Thiếu AutoModel mapping: " + model_type)
print(json.dumps({"transformers": transformers.__version__,
                  "architectures": [model_type for model_type, _ in required]}))
"""


def probe_transformers_models():
    loaded = sys.modules.get("transformers")
    if loaded is not None and loaded.__version__ != version("transformers"):
        raise RuntimeError("Kernel còn giữ Transformers cũ sau khi pip cài bản mới. "
                           "Restart kernel rồi chạy lại các cell từ đầu.")
    result = subprocess.run([sys.executable, "-c", TRANSFORMERS_COMPATIBILITY_CODE],
                            capture_output=True, text=True, check=False, timeout=60)
    if result.returncode != 0:
        raise RuntimeError("Transformers chưa hỗ trợ đủ kiến trúc PP-OCRv6 medium. "
                           f"Cài transformers=={TRANSFORMERS_VERSION}, restart kernel và chạy từ đầu.\n"
                           + (result.stderr or result.stdout)[-2500:])
    return json.loads(result.stdout.strip().splitlines()[-1])



In [ ]:
# Chạy trên phiên Kaggle/Colab mới; giữ nguyên PyTorch và bộ CUDA/NCCL sẵn có.
if installed_version("paddlepaddle-gpu") or installed_version("paddlepaddle"):
    raise RuntimeError("Phiên đã có PaddlePaddle. Lưu output rồi tạo phiên Kaggle/Colab mới trước khi chạy notebook này. "
                       "Không gỡ thủ công CUDA/NCCL trong phiên đang lỗi.")
torch_probe = probe_torch(DEVICE)
print("Runtime:", RUNTIME, "| Engine:", INFERENCE_ENGINE, "| PyTorch:", torch_probe)
stack_before = gpu_stack_versions()
install_dir = LOCAL_CACHE / "installation"
install_dir.mkdir(parents=True, exist_ok=True)
constraints = install_dir / "gpu-stack-constraints.txt"
constraints.write_text("".join(f"{name}=={v}\n" for name, v in sorted(stack_before.items())), encoding="utf-8")
packages = [f"paddleocr=={PADDLEOCR_VERSION}", f"paddlex[ocr-core]=={PADDLEX_VERSION}",
            f"transformers=={TRANSFORMERS_VERSION}", "google-api-python-client", "google-auth", "Pillow", "pandas", "tqdm", "einops>=0.8,<0.9", "gdown>=5.2,<6", "PyYAML>=6", "requests"]
command = [sys.executable, "-m", "pip", "install", "-c", str(constraints), *packages]
report_path = install_dir / "pip-plan.json"
# Xem kế hoạch trước: không cho pip thêm/hạ/nâng các gói GPU.
subprocess.check_call([*command, "--dry-run", "--report", str(report_path)])
reject_gpu_changes(json.loads(report_path.read_text(encoding="utf-8")))
subprocess.check_call(command)
# PyPI VietOCR 0.3.13 ghim Pillow 10.2 và các gói augmentation/training cũ.
# Chỉ cài wheel + phụ thuộc inference đã kiểm tra; không cài lại Torch/torchvision.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps",
                       f"vietocr=={VIETOCR_VERSION}"])
if gpu_stack_versions() != stack_before:
    raise RuntimeError("Bộ GPU đã thay đổi ngoài dự kiến. Dừng và tạo phiên mới trước khi OCR.")
probe_torch(DEVICE)
print("Kiểm tra kiến trúc:", probe_transformers_models())
# Kiểm tra import bằng tiến trình mới, chưa tải trọng số model.
subprocess.check_call([sys.executable, "-c",
    "from paddleocr import PaddleOCR; "
    "from transformers import AutoImageProcessor, AutoModelForObjectDetection, AutoModelForTextRecognition; "
    "from vietocr.tool.translate import build_model, process_input, translate; "
    "print('Import Paddle/Transformers/VietOCR OK; chua tai model.')"])
print("Đã cài SDK. Không cài Paddle GPU, không đổi Torch/CUDA/NCCL.")


## 4. Kết nối Drive

**Colab:** đăng nhập tài khoản được chia sẻ thư mục như trước.

**Kaggle:** không dùng `google.colab.drive.mount()` hoặc đăng nhập Colab trong kernel Kaggle.
Chạy helper `tao_quyen_drive.py` trên máy bạn để đăng nhập Google **một lần**,
rồi lưu nội dung token JSON vào **Kaggle Secrets**, tên `GOOGLE_DRIVE_CREDENTIALS`, bật quyền dùng cho notebook.
Notebook đọc Secret trong bộ nhớ, làm mới quyền OAuth và dùng Drive API đọc đúng các folder ID đã cấu hình.
Không dán token vào cell, output notebook hoặc chat. Quyền đã cấp có thể dùng lại qua Secret này ở phiên mới.

Nếu đã có service account: đặt `DRIVE_AUTH_MODE='service_account'`, dùng JSON của service account làm Secret và
chia sẻ thư mục nguồn cho `client_email` với quyền Viewer.

Kaggle lưu kết quả vào `/kaggle/working/NLP_OCR_TV02`, không tự đồng bộ lên Drive.
Với `INPUT_MODE='local'`, không cần Secret; sửa `local_folder` trỏ đến dữ liệu đã gắn vào notebook.


In [ ]:
drive_service = None
if RUNTIME == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
elif RUNTIME == "kaggle" and str(OUTPUT_ROOT).startswith("/content/drive"):
    raise ValueError("Kaggle không mount Drive như Colab. Đổi OUTPUT_ROOT sang /kaggle/working/NLP_OCR_TV02.")

if INPUT_MODE == "drive_api":
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    if RUNTIME == "colab":
        from google.colab import auth
        import google.auth
        auth.authenticate_user()
        credentials, _ = google.auth.default(scopes=[DRIVE_READ_SCOPE])
    elif RUNTIME == "kaggle":
        from kaggle_secrets import UserSecretsClient
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from google.oauth2 import service_account
        try:
            secret_value = UserSecretsClient().get_secret(DRIVE_SECRET_NAME)
        except Exception:
            raise RuntimeError(f"Thêm Kaggle Secret {DRIVE_SECRET_NAME} rồi bật quyền dùng cho notebook; xem phần 4 của notebook.") from None
        try:
            credential_info = json.loads(secret_value)
        except (TypeError, json.JSONDecodeError):
            raise ValueError("Kaggle Secret phải là nội dung JSON xác thực, không phải tên file hoặc đường dẫn.") from None
        finally:
            del secret_value
        validate_drive_credentials(credential_info, DRIVE_AUTH_MODE)
        try:
            if DRIVE_AUTH_MODE == "oauth":
                credentials = Credentials.from_authorized_user_info(credential_info, scopes=[DRIVE_READ_SCOPE])
            else:
                credentials = service_account.Credentials.from_service_account_info(
                    credential_info, scopes=[DRIVE_READ_SCOPE])
            credentials.refresh(Request())
        except Exception:
            raise RuntimeError("Không xác thực được Drive. Kiểm tra Secret/quyền đã cấp; nếu OAuth hết hạn hoặc bị thu hồi, đăng nhập lại bằng helper.") from None
        finally:
            del credential_info
    else:
        raise RuntimeError("Chọn INPUT_MODE='local' khi chạy local; hoặc mở notebook bằng Kaggle/Colab.")
    drive_service = build("drive", "v3", credentials=credentials, cache_discovery=False)
    print("Đã kết nối Drive bằng API đọc nguồn.")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
print("Kết quả:", RUN_ROOT)
if RUNTIME == "kaggle":
    print("Dùng Save Version hoặc tải ZIP để giữ kết quả trước khi kết thúc phiên Kaggle.")

## 5. Hàm kiểm kê nguồn — chạy cell, thường không cần sửa

Đọc đủ các trang kết quả Drive API, đi xuống thư mục con, lọc đúng TV/bài/trang.
Thư mục rỗng được bỏ qua. Hai file cùng đường dẫn và cùng MD5 chỉ nhận một lần;
khác nội dung thì dừng để bạn chọn lại nguồn, không ghi đè âm thầm.

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff", ".bmp"}
FOLDER_MIME = "application/vnd.google-apps.folder"
WORK_PATTERN = re.compile(r"^(CPO_\d+)_TV(\d+)_(.+)_(han|dich_am|dich_nghia|dich_tho)_v(\d+)$")

def safe_name(name):
    if not name or name in {".", ".."} or any(c in name for c in "/\\\x00"):
        raise ValueError(f"Tên file/thư mục không hợp lệ: {name!r}")
    return name

def natural_key(value):
    return tuple((1, int(x)) if x.isdigit() else (0, x.casefold())
                 for x in re.split(r"(\d+)", str(value)))

def list_drive_children(service, folder_id):
    token = None
    while True:
        response = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType,size,md5Checksum,modifiedTime)",
            pageSize=1000, pageToken=token, orderBy="name",
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute(num_retries=3)
        yield from response.get("files", [])
        token = response.get("nextPageToken")
        if not token:
            break

def walk_drive(service, folder_id):
    stack, visited = [(folder_id, Path())], set()
    while stack:
        current, relative = stack.pop()
        if current in visited:
            continue
        visited.add(current)
        for file in list_drive_children(service, current):
            child = relative / safe_name(file["name"])
            if file["mimeType"] == FOLDER_MIME:
                stack.append((file["id"], child))
            else:
                yield {**file, "relative_path": child.as_posix(), "parent_id": current}

def walk_local(folder):
    folder = folder.resolve()
    if not folder.is_dir():
        raise FileNotFoundError(f"Không thấy thư mục: {folder}")
    for path in folder.rglob("*"):
        if path.is_file():
            if not path.resolve().is_relative_to(folder):
                raise ValueError(f"File trỏ ra ngoài thư mục nguồn: {path}")
            yield {"id": str(path.resolve()), "name": path.name,
                   "relative_path": path.relative_to(folder).as_posix(),
                   "local_path": str(path.resolve()), "size": path.stat().st_size}

def within_scope(file, job):
    relative = Path(file["relative_path"])
    extension = relative.suffix.lower()
    if extension not in (IMAGE_EXTENSIONS if job["mode"] == "ocr" else {".txt"}):
        return None, "định dạng khác"
    candidates = [relative.stem] + list(reversed(relative.parts[:-1]))
    matches = [(s, WORK_PATTERN.fullmatch(s)) for s in candidates]
    named = next(((s, m) for s, m in matches if m), None)
    if named:
        stem, match = named
        code_value, member, work, kind, _ = match.groups()
        if job.get("code") != code_value or job["kind"] != kind:
            return None, "khác tác phẩm/phiên bản"
        if job.get("member") is not None and int(member) != job["member"]:
            return None, "khác thành viên"
    else:
        if job.get("member") is not None or job.get("work_range") is not None:
            return None, "không xác định được mã TV/bài từ tên nguồn"
        stem, work = relative.parent.name or relative.stem, relative.parent.name or relative.stem
    work_match = re.fullmatch(r"bai(\d+)", work)
    if job.get("work_range"):
        low, high = job["work_range"]
        if not work_match or not low <= int(work_match[1]) <= high:
            return None, "ngoài khoảng bài"
    only = ONLY_WORKS.get(job["code"], [])
    if only and work not in only:
        return None, "không nằm trong ONLY_WORKS"
    page_match = re.search(r"pdfp(\d+)", relative.stem)
    page = int(page_match[1]) if page_match else None
    if job.get("page_range"):
        low, high = job["page_range"]
        if page is None or not low <= page <= high:
            return None, "ngoài khoảng trang hoặc không xác định được trang"
    if page in job.get("exclude_pages", []):
        return None, "trang đã được yêu cầu bỏ khỏi ảnh nộp"
    return {"work_stem": safe_name(stem), "work": work, "page": page}, None

def deduplicate_files(files):
    unique = {}
    for file in files:
        key = file["relative_path"]
        if key in unique:
            old = unique[key]
            same_bytes = (old.get("md5Checksum") and old["md5Checksum"] == file.get("md5Checksum"))
            if old["id"] == file["id"] or same_bytes:
                old.setdefault("duplicate_ids", []).append(file["id"])
                continue
            raise ValueError(f"Hai file khác nội dung cùng đường dẫn {key}: {old['id']} và {file['id']}. Chọn thư mục cụ thể hơn.")
        unique[key] = dict(file)
    return sorted(unique.values(), key=lambda x: natural_key(x["relative_path"]))

## 6. Kiểm kê phần đã chọn

Cell này chưa OCR. Bảng thống kê là **số ảnh/vùng đã cắt**, không phải số trang PDF gốc.
Các file không đúng bộ lọc được ghi trong `excluded_files.json` để kiểm tra, không tự xóa.
Nếu một nguồn không có file hợp lệ, dừng để sửa cấu hình.

In [ ]:
from IPython.display import display
import pandas as pd

all_items, excluded_files = [], []
for key in SELECTED_JOBS:
    job = JOBS[key]
    if INPUT_MODE == "drive_api":
        raw_files = walk_drive(drive_service, job["folder_id"])
    else:
        local_folder = Path(job["local_folder"])
        raw_files = walk_local(local_folder if local_folder.is_absolute() else LOCAL_INPUT_ROOT / local_folder)
    files = deduplicate_files(raw_files)
    accepted = []
    for file in files:
        scope, reason = within_scope(file, job)
        if reason:
            excluded_files.append({"job": key, "file": file["relative_path"], "reason": reason})
        else:
            accepted.append({**file, **scope, "job": key, "mode": job["mode"]})
    if not accepted:
        raise ValueError(f"{key}: không có file hợp lệ. Kiểm tra folder_id, member, work_range và page_range.")
    all_items.extend(accepted)

# Tên ảnh không được gây đụng file đầu ra của hai nguồn khác nhau.
seen_outputs = {}
for item in all_items:
    job = JOBS[item["job"]]
    output_key = (job["code"], job["kind"], item["mode"], item["work_stem"], Path(item["name"]).stem)
    if output_key in seen_outputs:
        raise ValueError(f"Đụng tên đầu ra: {item['relative_path']} và {seen_outputs[output_key]}. Chọn lại thư mục hoặc phiên bản.")
    seen_outputs[output_key] = item["relative_path"]

selected_items, job_counts = [], Counter()
for item in all_items:
    if item["mode"] == "ocr":
        if MAX_IMAGES_PER_JOB is not None and job_counts[item["job"]] >= MAX_IMAGES_PER_JOB:
            continue
        job_counts[item["job"]] += 1
    selected_items.append(item)

display(pd.DataFrame([
    {"nguon": key, "che_do": JOBS[key]["mode"],
     "tong_file_hop_le": sum(i["job"] == key for i in all_items),
     "file_se_chay": sum(i["job"] == key for i in selected_items),
     "so_tac_pham_co_file": len({i["work_stem"] for i in all_items if i["job"] == key})}
    for key in SELECTED_JOBS
]))
print("Tổng file hợp lệ:", len(all_items), "| file sẽ chạy:", len(selected_items))
display(pd.DataFrame([{k: i.get(k) for k in ["job", "relative_path", "work", "page"]}
                      for i in selected_items[:12]]))

## 7. Hàm lưu và đọc nguồn

Ảnh được tải vào cache tạm của runtime bằng ID, kiểm tra MD5 nếu Drive cung cấp.
TXT nguồn được sao chép nguyên byte. Kết quả có hash nguồn và phiên bản model để kiểm tra khi chạy tiếp.

In [ ]:
def bytes_hash(data, algorithm="sha256"):
    return hashlib.new(algorithm, data).hexdigest()

def file_hash(path, algorithm="sha256"):
    digest = hashlib.new(algorithm)
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, indent=2) + "\n").encode("utf-8")

def atomic_write(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".part")
    try:
        temp.write_bytes(data)
        temp.replace(path)
    finally:
        temp.unlink(missing_ok=True)

def input_path(item):
    if "local_path" in item:
        return Path(item["local_path"])
    path = LOCAL_CACHE / item["job"] / safe_name(item["id"]) / safe_name(item["name"])
    if path.is_file() and item.get("md5Checksum") and file_hash(path, "md5") == item["md5Checksum"]:
        return path
    # Có thể tải lại cache chưa có checksum; không dùng cache cũ mà không kiểm chứng.
    request = drive_service.files().get_media(fileId=item["id"], supportsAllDrives=True)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request, chunksize=8 * 1024 * 1024)
    done = False
    while not done:
        _, done = downloader.next_chunk(num_retries=3)
    data = buffer.getvalue()
    if item.get("md5Checksum") and bytes_hash(data, "md5") != item["md5Checksum"]:
        raise OSError(f"Checksum không khớp: {item['relative_path']}")
    if item.get("size") is not None and len(data) != int(item["size"]):
        raise OSError(f"Dung lượng không khớp: {item['relative_path']}")
    atomic_write(path, data)
    return path

def output_paths(item):
    job = JOBS[item["job"]]
    side = Path("han") if job["kind"] == "han" else Path("viet") / job["kind"]
    if item["mode"] == "text":
        base = RUN_ROOT / job["code"] / "01_nguon" / side
        txt = base / safe_name(item["name"])
    else:
        base = RUN_ROOT / job["code"] / "02_ocr_tho" / side / item["work_stem"]
        txt = base / (Path(item["name"]).stem + ".txt")
    result = {"txt": txt, "meta": txt.with_suffix(".meta.json")}
    if item["mode"] == "ocr":
        result["json"] = txt.with_suffix(".json")
        if item["job"] == "bac_su_han":
            result["image"] = txt.with_name(txt.stem + ".input.png")
    for path in result.values():
        if not path.resolve().is_relative_to(RUN_ROOT.resolve()):
            raise ValueError(f"Đường dẫn đầu ra nằm ngoài RUN_ROOT: {path}")
        if "local_path" in item and path.resolve() == Path(item["local_path"]).resolve():
            raise ValueError("Đường dẫn đầu ra trùng nguồn; đổi OUTPUT_ROOT.")
    return result

report_dir = RUN_ROOT / "05_quan_ly"
report_dir.mkdir(parents=True, exist_ok=True)
atomic_write(report_dir / "inventory.json", json_bytes({"items": all_items, "selected": selected_items}))
atomic_write(report_dir / "excluded_files.json", json_bytes(excluded_files))

## 8. Nạp model theo nguồn đã chọn

Cell helper chuẩn bị cách tải config/checkpoint VietOCR có cache và hash; cell sau nạp model.
- Hán: PaddleOCR detection + recognition, Transformers/PyTorch.
- Ảnh dịch âm/nghĩa/thơ: Paddle TextDetection → cắt/nắn dòng trong bộ nhớ → VietOCR.
- TXT Bắc sứ Việt: giữ nguyên byte, không nạp model nếu chỉ chọn nguồn này.

Chỉ nạp nhánh có trong `selected_items`. Hai nhánh dùng cùng kiến trúc detection; nhánh Hán giữ pipeline Paddle hiện có.
Config VietOCR lấy từ commit cố định, checkpoint chính thức hoặc `VIETOCR_WEIGHTS` do bạn chỉ định.


Nhánh Hán dùng `HAN_LAYOUTS` ở cell 1: Thi Đình/Bắc sứ theo cột dọc phải→trái,
Lý–Trần theo hàng ngang trái→phải. Chạy cả helper **bố cục Hán** và **tách hàng Hán**
trước cell nạp model. Lý–Trần nạp `TextRecognition` v6, không dùng detector cột dọc.
Chỉ nạp các nhánh có ảnh trong danh sách kiểm kê; vẫn giữ v6 + batch size 1.


In [ ]:
def cached_model_asset(url, filename):
    import requests

    directory = LOCAL_CACHE / "models" / "vietocr" / bytes_hash(url.encode())[:12]
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / filename
    receipt = target.with_suffix(target.suffix + ".sha256")
    if target.is_file() and receipt.is_file():
        if file_hash(target) != receipt.read_text(encoding="ascii").strip():
            raise OSError(f"Cache model đã thay đổi: {target}. Chọn LOCAL_CACHE mới.")
        return target
    part = target.with_suffix(target.suffix + ".part")
    with requests.get(url, stream=True, timeout=(15, 120)) as response:
        response.raise_for_status()
        received = 0
        with part.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
                    received += len(chunk)
        expected = response.headers.get("Content-Length")
        if expected and not response.headers.get("Content-Encoding") and received != int(expected):
            raise OSError(f"Tải model chưa đủ byte: {url}")
    if not received:
        raise OSError(f"File model/config rỗng: {url}")
    os.replace(part, target)
    receipt.write_text(file_hash(target), encoding="ascii")
    return target


def load_vietocr_model():
    import torch
    import yaml
    from vietocr.tool.translate import build_model, process_input, translate

    profiles = {"vgg_transformer": "vgg-transformer.yml", "vgg_seq2seq": "vgg-seq2seq.yml"}
    base_url = f"https://raw.githubusercontent.com/pbcquoc/vietocr/{VIETOCR_CONFIG_REVISION}/config/"
    configs = []
    for filename in ("base.yml", profiles[VIETOCR_MODEL]):
        config_file = cached_model_asset(base_url + filename, filename)
        data = yaml.safe_load(config_file.read_text(encoding="utf-8"))
        if not isinstance(data, dict):
            raise ValueError("Config VietOCR không phải YAML dạng dictionary.")
        configs.append(data)
    config = {**configs[0], **configs[1]}
    config["cnn"]["pretrained"] = False  # Đã dùng checkpoint OCR, không tải ImageNet.
    config["device"] = "cuda:" + DEVICE.split(":")[-1] if ":" in DEVICE else ("cuda:0" if DEVICE == "gpu" else "cpu")
    config["dataset"]["image_max_width"] = VIETOCR_MAX_WIDTH
    config["predictor"]["beamsearch"] = False
    if VIETOCR_WEIGHTS:
        weights = Path(VIETOCR_WEIGHTS).expanduser()
        if not weights.is_file():
            raise FileNotFoundError(f"Không có checkpoint VietOCR: {weights}")
    else:
        url = config["weights"]
        if not url.startswith("https://"):
            raise ValueError("Checkpoint mặc định phải dùng HTTPS.")
        weights = cached_model_asset(url, VIETOCR_MODEL + ".pth")

    # Adapter nhỏ dùng chính build_model/translate của VietOCR 0.3.13.
    # Chỉ nạp state_dict, không dùng weights_only=False hay sửa torch.load toàn cục.
    class VietLineReader:
        def __init__(self):
            self.model, self.vocab = build_model(config)
            state = torch.load(str(weights), map_location="cpu", weights_only=True)
            self.model.load_state_dict(state)
            self.model.eval()
            self.config = config

        def prepare_input(self, image, max_width=None):
            width = VIETOCR_MAX_WIDTH if max_width is None else max_width
            if not isinstance(width, int) or width < 32:
                raise ValueError("max_width phải là số nguyên >= 32.")
            # Đây là cùng hàm preprocessing dùng cho OCR hàng loạt.
            return process_input(image, config["dataset"]["image_height"],
                                 config["dataset"]["image_min_width"], width)

        def predict_prepared(self, batch, max_length=None, return_trace=False):
            limit = VIETOCR_MAX_LENGTH if max_length is None else max_length
            if not isinstance(limit, int) or limit <= 0:
                raise ValueError("max_length phải là số nguyên dương.")
            # Nhận đúng tensor đã xem ở diagnostic, không preprocessing lần hai.
            model_input = batch.to(config["device"])
            tokens, probabilities = translate(model_input, self.model, max_seq_length=limit)
            ids = [int(token) for token in tokens[0].tolist()]
            text = self.vocab.decode(ids)
            probability = float(probabilities[0])
            if not return_trace:
                return text, probability
            eos = int(self.vocab.eos)
            trace = {"input_shape": list(batch.shape), "input_dtype": str(batch.dtype),
                     "token_ids": ids, "generated_tokens": max(0, len(ids) - 1),
                     "eos_index": ids.index(eos) if eos in ids else None,
                     "stop_reason": "eos" if eos in ids else "length_limit",
                     "decode_limit": limit}
            return text, probability, trace

        def predict(self, image, return_prob=True):
            text, probability = self.predict_prepared(self.prepare_input(image))
            return (text, probability) if return_prob else text

    identity = {"weights_sha256": file_hash(weights),
                "config_sha256": bytes_hash(json_bytes(config)),
                "config_revision": VIETOCR_CONFIG_REVISION}
    return VietLineReader(), identity


In [ ]:
def han_policy(item):
    """Trả cấu hình đúng nguồn/ảnh để cả inference và fingerprint dùng chung."""
    if item["job"] not in HAN_LAYOUTS:
        raise ValueError("Chưa khai báo HAN_LAYOUTS cho nguồn Hán: " + item["job"])
    key = item["job"] + "/" + item["relative_path"]
    override = HAN_IMAGE_OVERRIDES.get(key, {})
    if set(override) - {"layout", "row_settings", "row_boxes", "order_indices"}:
        raise ValueError("Khóa ngoại lệ Hán không hợp lệ: " + key)
    layout = override.get("layout", HAN_LAYOUTS[item["job"]])
    if layout not in {"vertical_rtl", "horizontal_ltr"}:
        raise ValueError("Chiều đọc Hán không hợp lệ: " + str(layout))
    result = {"layout": layout,
              "reading_order": "top_to_bottom" if layout == "horizontal_ltr" else "vertical_rtl",
              "order_indices": override.get("order_indices")}
    if item["job"] == "bac_su_han" and layout == "vertical_rtl":
        result.update(reading_order="vertical_columns_rtl", column_center_ratio=BAC_COLUMN_CENTER_RATIO)
    if layout == "horizontal_ltr":
        settings = {**HAN_ROW_SETTINGS, **override.get("row_settings", {})}
        if set(settings) != set(HAN_ROW_SETTINGS):
            raise ValueError("Tham số tách hàng không hợp lệ: " + key)
        result.update(row_settings=settings, row_boxes=override.get("row_boxes"))
    elif "row_boxes" in override or "row_settings" in override:
        raise ValueError("row_boxes/row_settings chỉ dùng cho hàng ngang.")
    return result


def checked_order(indices, count):
    indices = list(indices)
    if any(type(i) is not int for i in indices) or sorted(indices) != list(range(count)):
        raise ValueError("Thứ tự phải chứa đủ mỗi ID vùng đúng một lần; không bỏ/xóa vùng.")
    return indices


def han_geometry_review(data, policy):
    """Chỉ cảnh báo hình học; không bỏ vùng, sửa chữ hoặc chứng nhận bố cục."""
    import statistics
    polygons = data.get("rec_polys", [])
    bounds = [(min(p[0] for p in points), min(p[1] for p in points),
               max(p[0] for p in points), max(p[1] for p in points)) for points in polygons]
    heights = [max(1, b[3]-b[1]) for b in bounds]
    areas = [max(1, b[2]-b[0])*height for b, height in zip(bounds, heights)]
    median_h = statistics.median(heights) if heights else 0
    median_a = statistics.median(areas) if areas else 0
    small = [i for i, (height, area) in enumerate(zip(heights, areas))
             if height < 0.35*median_h or area < 0.15*median_a]
    separated = []
    if policy["layout"] == "vertical_rtl":
        for i, a in enumerate(bounds):
            for j in range(i+1, len(bounds)):
                b = bounds[j]
                overlap = max(0, min(a[2], b[2])-max(a[0], b[0]))
                narrow = max(1, min(a[2]-a[0], b[2]-b[0]))
                gap = max(b[1]-a[3], a[1]-b[3])
                if overlap/narrow > 0.4 and gap > 0.7*narrow:
                    separated.append([i, j])
    flags = list(data.get("row_detection", {}).get("flags", []))
    if small: flags.append("small_or_short_regions")
    if separated: flags.append("separated_regions_in_column")
    return {"flags": flags, "small_region_indices": small, "separated_region_pairs": separated}


def grouped_vertical_columns(data, center_ratio):
    """Gom cột bằng tâm X cố định của vùng cao; tránh nối dây chuyền giữa cột kề nhau."""
    import statistics
    import math
    texts, polygons = data.get("rec_texts", []), data.get("rec_polys", [])
    if len(polygons) != len(texts):
        raise ValueError("Không đủ tọa độ để nhóm cột.")
    if not 0 < center_ratio < 1:
        raise ValueError("Dung sai nhóm cột phải nằm giữa 0 và 1.")
    if not texts:
        return {"order": [], "groups": [], "ambiguous_indices": [], "tolerance": 0.0}
    bounds = []
    for points in polygons:
        if not points or any(not math.isfinite(float(v)) for point in points for v in point):
            raise ValueError("Tọa độ cột không hợp lệ.")
        x1, x2 = min(p[0] for p in points), max(p[0] for p in points)
        y1, y2 = min(p[1] for p in points), max(p[1] for p in points)
        bounds.append((float(x1), float(y1), float(x2), float(y2)))
    centers = [(b[0]+b[2])/2 for b in bounds]
    widths = [max(1.0, b[2]-b[0]) for b in bounds]
    heights = [max(1.0, b[3]-b[1]) for b in bounds]
    tall = [i for i in range(len(texts)) if heights[i] >= 2*widths[i]]
    typical = statistics.median([widths[i] for i in tall] or widths)
    tolerance = typical * center_ratio
    anchors, groups, ambiguous = [], [], []
    # Vùng cao trước để chữ/số nhỏ không kéo tâm cột và nối nhầm hai cột.
    candidates = sorted(tall, key=lambda i: (-heights[i], -centers[i], i))
    candidates += sorted(set(range(len(texts)))-set(tall), key=lambda i: (-centers[i], bounds[i][1], i))
    for index in candidates:
        near = [(abs(centers[index]-x), group) for group, x in enumerate(anchors)
                if abs(centers[index]-x) <= tolerance]
        if near:
            if len(near) > 1: ambiguous.append(index)
            group = min(near)[1]
            groups[group].append(index)
        else:
            anchors.append(centers[index]); groups.append([index])
    columns = [sorted(groups[g], key=lambda i: (bounds[i][1], -centers[i], i))
               for g in sorted(range(len(groups)), key=lambda g: (-anchors[g], g))]
    return {"order": [i for column in columns for i in column], "groups": columns,
            "ambiguous_indices": ambiguous, "tolerance": tolerance}


In [ ]:
def horizontal_han_rows(image, settings, manual_boxes=None):
    """Tìm dải hàng ngang; không sửa/lọc pixel ảnh đưa vào recognition."""
    import math
    import statistics
    from PIL import ImageFilter

    width, height = image.size
    if manual_boxes is not None:
        boxes = []
        for box in manual_boxes:
            if len(box) != 4 or any(type(v) is not int for v in box):
                raise ValueError("row_boxes cần bốn tọa độ pixel nguyên cho mỗi hàng.")
            x1, y1, x2, y2 = box
            if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
                raise ValueError("row_boxes nằm ngoài ảnh hoặc có diện tích bằng 0.")
            boxes.append(list(box))
        if not boxes:
            raise ValueError("row_boxes không được rỗng; bỏ khóa này để tự tách hàng.")
        return boxes, {"method": "manual_rows", "source_size": [width, height], "flags": []}
    if (settings["max_side"] < 64 or not 0 < settings["ink_threshold"] < 256
        or not 0 < settings["min_ink_fraction"] < 1 or settings["bridge_gap"] < 0
        or not 0 < settings["min_height_fraction"] < 1 or settings["padding_fraction"] < 0
        or settings["min_padding_fraction"] < 0):
        raise ValueError("Tham số tách hàng ngoài phạm vi hợp lệ.")
    analysis = image.convert("L")
    analysis.thumbnail((settings["max_side"], settings["max_side"]))
    aw, ah = analysis.size
    # Median chỉ loại chấm nhiễu cho việc xác định khoảng trắng giữa hàng.
    mask = analysis.filter(ImageFilter.MedianFilter(3)).point(
        [255 if value < settings["ink_threshold"] else 0 for value in range(256)])
    pixels = mask.tobytes()
    ink = [pixels[y*aw:(y+1)*aw].count(255) for y in range(ah)]
    minimum = max(3, math.ceil(aw * settings["min_ink_fraction"]))
    bands, start, last = [], None, None
    for y, count in enumerate(ink):
        if count < minimum:
            continue
        if start is None:
            start = y
        elif y - last - 1 > settings["bridge_gap"]:
            bands.append((start, last + 1))
            start = y
        last = y
    if start is not None:
        bands.append((start, last + 1))
    minimum_height = max(4, math.ceil(aw * settings["min_height_fraction"]))
    rejected = [list(b) for b in bands if b[1] - b[0] < minimum_height]
    bands = [b for b in bands if b[1] - b[0] >= minimum_height]
    boxes = []
    for i, (top, bottom) in enumerate(bands):
        # Giới hạn tại khoảng trắng giữa hai hàng, có lề giữ nét mảnh/dấu câu.
        upper = (bands[i-1][1] + top)//2 if i else 0
        lower = (bottom + bands[i+1][0])//2 if i+1 < len(bands) else ah
        padding = max(2, math.ceil(aw * settings["min_padding_fraction"]),
                      math.ceil((bottom-top) * settings["padding_fraction"]))
        y1 = max(upper, top-padding)
        y2 = min(lower, bottom+padding)
        boxes.append([0, math.floor(y1 * height/ah), width, math.ceil(y2 * height/ah)])
    flags = []
    if not boxes:
        flags.append("no_horizontal_rows")
    median_height = statistics.median([b[1]-b[0] for b in bands]) if bands else 0
    tall = [i for i, (a, z) in enumerate(bands) if z-a > 1.8 * median_height]
    if tall:
        flags.append("unusual_row_height")
    covered = set()
    for _, y1, _, y2 in boxes:
        covered.update(range(max(0, math.floor(y1*ah/height)), min(ah, math.ceil(y2*ah/height))))
    total_ink = sum(ink)
    outside = sum(value for y, value in enumerate(ink) if y not in covered)
    outside_fraction = outside/total_ink if total_ink else 0.0
    if outside_fraction > 0.02:
        flags.append("ink_outside_rows")
    return boxes, {"method": "horizontal_projection_v1", "source_size": [width, height],
        "analysis_size": [aw, ah], "settings": settings, "bands": [list(b) for b in bands],
        "rejected_short_bands": rejected, "unusual_row_indices": tall,
        "outside_ink_fraction": outside_fraction, "flags": flags}


def predict_horizontal_han(path, policy):
    import math
    import tempfile
    from PIL import Image

    if han_row_recognizer is None:
        raise RuntimeError("Chưa nạp recognition Hán hàng ngang. Chạy lại mục 8.")
    with Image.open(path) as original:
        image = original.convert("RGB")
    boxes, detection = horizontal_han_rows(image, policy["row_settings"], policy["row_boxes"])
    texts, scores, polygons, details, originals = [], [], [], [], []
    LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="han_rows_", dir=LOCAL_CACHE) as scratch:
        for index, box in enumerate(boxes):
            crop = image.crop(box)
            filename = Path(scratch) / f"row_{index:04d}.png"
            crop.save(filename, format="PNG", compress_level=0)
            predictions = list(han_row_recognizer.predict(input=str(filename), batch_size=HAN_REC_BATCH_SIZE))
            if len(predictions) != 1:
                raise ValueError("Một hàng Hán phải có đúng một kết quả recognition.")
            raw = predictions[0].json
            raw = json.loads(raw) if isinstance(raw, str) else raw
            result = raw.get("res", raw)
            text = result["rec_text"]
            if not isinstance(text, str):
                raise TypeError("rec_text phải là chuỗi.")
            score = result.get("rec_score")
            score = float(score) if score is not None and math.isfinite(float(score)) else None
            x1, y1, x2, y2 = box
            polygon = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]
            flags = ["unusual_row_height"] if index in detection.get("unusual_row_indices", []) else []
            if not text.strip(): flags.append("empty_line")
            if score is None: flags.append("unknown_confidence")
            texts.append(text); scores.append(score); polygons.append(polygon); originals.append(raw)
            details.append({"detection_index": index, "crop_box": box, "crop_size": list(crop.size),
                "crop_rgb_sha256": bytes_hash(crop.tobytes()), "flags": flags})
    return {"res": {"input_path": str(path), "pipeline": "paddle_han_horizontal_rows",
        "coordinate_space": "source_image_pixels", "rec_texts": texts, "rec_scores": scores,
        "rec_polys": polygons, "line_details": details, "row_detection": detection,
        "recognizer_results": originals}}


In [ ]:
if "selected_items" not in globals():
    raise RuntimeError("Chưa có selected_items. Chạy cấu hình → kết nối → kiểm kê → hàm đọc/lưu nguồn trước mục 8.")
ocr = han_row_recognizer = viet_detector = viet_recognizer = None
viet_model_identity = {}
runtime_versions = {"python": sys.version.split()[0]}
has_han = any(i["mode"] == "ocr" and JOBS[i["job"]]["kind"] == "han" for i in selected_items)
has_viet = any(i["mode"] == "ocr" and JOBS[i["job"]]["kind"] != "han" for i in selected_items)
has_han_columns = any(i["mode"] == "ocr" and JOBS[i["job"]]["kind"] == "han"
    and han_policy(i)["layout"] == "vertical_rtl" for i in selected_items)
has_han_rows = any(i["mode"] == "ocr" and JOBS[i["job"]]["kind"] == "han"
    and han_policy(i)["layout"] == "horizontal_ltr" for i in selected_items)
if has_han or has_viet:
    torch_probe = probe_torch(DEVICE)
    print("Kiểm tra kiến trúc:", probe_transformers_models())
    from paddleocr import PaddleOCR, TextDetection

    runtime_versions.update({p: version(p) for p in ["paddleocr", "paddlex", "torch", "torchvision", "transformers", "Pillow"]})
    runtime_versions["cuda"] = torch_probe["cuda"]
if has_han_columns:
    ocr = PaddleOCR(
        text_detection_model_name=DETECTION_MODEL,
        text_recognition_model_name=RECOGNITION_MODEL,
        text_recognition_batch_size=HAN_REC_BATCH_SIZE,
        engine=INFERENCE_ENGINE, device=DEVICE,
        use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
    )
    print("Hán cột dọc: Paddle detection + recognition trên", DEVICE)
if has_han_rows:
    from paddleocr import TextRecognition
    han_row_recognizer = TextRecognition(model_name=RECOGNITION_MODEL,
        engine=INFERENCE_ENGINE, device=DEVICE)
    print("Hán hàng ngang: tách hàng từ ảnh nguồn →", RECOGNITION_MODEL, "batch", HAN_REC_BATCH_SIZE)
if has_viet:
    viet_detector = TextDetection(model_name=DETECTION_MODEL, engine=INFERENCE_ENGINE, device=DEVICE)
    viet_recognizer, viet_model_identity = load_vietocr_model()
    runtime_versions.update({p: version(p) for p in ["vietocr", "einops"]})
    print("Việt: Paddle detection + VietOCR", VIETOCR_MODEL, "trên", DEVICE)
print(runtime_versions)


## 9. Hàm cắt dòng, nhận dạng và lưu kết quả

Cell đầu: nắn tứ giác detection thành ảnh dòng Việt, chỉ trong bộ nhớ; không sinh ảnh/PDF nguồn mới.
Cell sau: chọn nhánh bằng `JOBS[job]["kind"]`, nhận dạng và bảo vệ kết quả đã có.
Hán giữ JSON Paddle gốc. Việt lưu JSON tổng hợp gồm detector_result gốc, chữ/điểm VietOCR,
tọa độ, vùng crop và cảnh báo dòng quá dài. TXT chỉ gom dự đoán, chưa sửa chữ/làm sạch.

`VIET_READING_ORDER="top_to_bottom"` dành cho vùng một cột; trang nhiều cột cần xem thử.
Điểm Paddle/VietOCR không được xem là cùng một thang đo đã hiệu chuẩn.


Chạy các helper tiền xử lý Bắc sứ trong mục 9 trước `ocr_helpers`; 10b/11 tự dùng cấu hình chính.
Ảnh đầu vào thực được cache và lưu cạnh TXT với đuôi `.input.png` khi chạy 11.
Tọa độ giữ theo pixel ảnh nguồn; metadata ghi mode/tham số/hash cùng các nhóm cột.


In [ ]:
def line_crop(image, polygon, padding=0):
    """Nắn tứ giác TL/TR/BR/BL thành ảnh dòng; chỉ thao tác trong bộ nhớ."""
    import math
    from PIL import Image

    if len(polygon) != 4 or any(len(p) != 2 for p in polygon):
        raise ValueError("Vùng dòng phải có đúng bốn đỉnh (x, y).")
    points = [(float(x), float(y)) for x, y in polygon]
    if not all(math.isfinite(v) for p in points for v in p):
        raise ValueError("Tọa độ dòng không hữu hạn.")
    # Detector trả đỉnh theo TL, TR, BR, BL.
    tl, tr, br, bl = points
    distance = lambda a, b: math.hypot(a[0] - b[0], a[1] - b[1])
    width, height = max(distance(tl, tr), distance(bl, br)), max(distance(tl, bl), distance(tr, br))
    area = abs(sum(points[i][0] * points[(i+1) % 4][1] - points[(i+1) % 4][0] * points[i][1] for i in range(4))) / 2
    if width < 1 or height < 1 or area < 1 or any(distance(points[i], points[(i+1) % 4]) < 1 for i in range(4)):
        raise ValueError("Vùng dòng rỗng hoặc suy biến; giữ lỗi để rà soát ảnh.")
    ux, uy = (tr[0] - tl[0]) / distance(tl, tr), (tr[1] - tl[1]) / distance(tl, tr)
    vx, vy = (bl[0] - tl[0]) / distance(tl, bl), (bl[1] - tl[1]) / distance(tl, bl)
    padded = [(x + sx * padding * ux + sy * padding * vx,
               y + sx * padding * uy + sy * padding * vy)
              for (x, y), sx, sy in zip(points, [-1, 1, 1, -1], [-1, -1, 1, 1])]
    # Pillow QUAD cần thứ tự TL, BL, BR, TR.
    quad = tuple(v for i in [0, 3, 2, 1] for v in padded[i])
    size = (math.ceil(width + 2 * padding), math.ceil(height + 2 * padding))
    source = image if image.mode == "RGB" else image.convert("RGB")
    crop = source.transform(size, Image.Transform.QUAD, quad,
                                          resample=Image.Resampling.BICUBIC, fillcolor="white")
    return crop, padded


def detect_viet_regions(path):
    if viet_detector is None:
        raise RuntimeError("Chưa nạp detector nhánh Việt ở mục 8.")
    results = list(viet_detector.predict(input=str(path), batch_size=1,
        limit_side_len=DETECTION_MAX_SIDE, limit_type="max",
        box_thresh=DETECTION_BOX_THRESHOLD, unclip_ratio=DETECTION_UNCLIP_RATIO))
    if len(results) != 1:
        raise ValueError("Mỗi ảnh phải có đúng một kết quả detection.")
    raw = results[0].json
    data = raw.get("res", raw)
    if len(data["dt_polys"]) != len(data["dt_scores"]):
        raise ValueError("Số vùng và điểm detection không khớp.")
    return raw, data


def predict_viet_image(path):
    import math
    from PIL import Image

    if viet_recognizer is None:
        raise RuntimeError("Chưa nạp VietOCR ở mục 8.")
    raw_detection, detection = detect_viet_regions(path)
    texts, scores, details = [], [], []
    with Image.open(path) as original:
        image = original.convert("RGB")
    for index, polygon in enumerate(detection["dt_polys"]):
        crop, crop_quad = line_crop(image, polygon, VIET_CROP_PADDING)
        text, score = viet_recognizer.predict(crop, return_prob=True)
        score = float(score) if score is not None and math.isfinite(float(score)) else None
        flags = []
        if crop.width * 32 / crop.height > VIETOCR_MAX_WIDTH:
            flags.append("width_clamped")
        if len(text) >= VIETOCR_MAX_LENGTH:
            flags.append("possible_decode_limit")
        if score is None:
            flags.append("unknown_confidence")
        if not text.strip():
            flags.append("empty_line")
        texts.append(text)
        scores.append(score)
        details.append({"detection_index": index, "crop_quad": crop_quad,
                        "crop_size": list(crop.size), "flags": flags})
    # Mảng giữ nguyên ID detection. Thứ tự TXT được lưu riêng, không sửa dữ liệu gốc.
    return {"res": {"input_path": str(path), "pipeline": "paddle_detection_vietocr",
        "rec_texts": texts, "rec_scores": scores, "rec_polys": detection["dt_polys"],
        "dt_scores": detection["dt_scores"], "line_details": details, "detector_result": raw_detection}}


def display_detection_image(path, polygons, order_indices=None):
    from PIL import Image, ImageDraw
    from IPython.display import Image as NotebookImage, display

    with Image.open(path) as original:
        preview = original.convert("RGB")
    width, height = preview.size
    preview.thumbnail((1100, 1500))
    sx, sy = preview.width / width, preview.height / height
    draw = ImageDraw.Draw(preview)
    labels = ({index: str(rank+1) for rank, index in enumerate(checked_order(order_indices, len(polygons)))}
              if order_indices is not None else {index: str(index) for index in range(len(polygons))})
    for index, polygon in enumerate(polygons):
        points = [(int(x * sx), int(y * sy)) for x, y in polygon]
        draw.line(points + [points[0]], fill="red", width=2)
        draw.text(points[0], labels[index], fill="blue")
    buffer = io.BytesIO()
    preview.save(buffer, format="JPEG", quality=85)
    display(NotebookImage(data=buffer.getvalue(), format="jpeg"))


In [ ]:
def bac_su_preprocess(image, params):
    """Ảnh dẫn xuất để thử OCR; giữ nguyên kích thước và pixel ảnh nguồn."""
    from PIL import Image, ImageOps, ImageChops, ImageFilter, ImageMath

    expected = {"red_hue_width", "red_min_saturation", "red_min_r_minus_g",
        "red_protect_max_below", "background_max_side", "background_filter_size",
        "background_blur", "black_point", "white_point", "tone_gamma", "binary_threshold"}
    if set(params) != expected:
        raise ValueError("Thiếu/thừa khóa BAC_PREPROCESS_PARAMS: " + str(set(params) ^ expected))
    if (not 0 <= params["red_hue_width"] <= 40
        or not 0 <= params["red_min_saturation"] <= 255
        or not 0 <= params["red_min_r_minus_g"] <= 255
        or not 0 <= params["red_protect_max_below"] <= 255
        or params["background_max_side"] < 32
        or params["background_filter_size"] < 3 or params["background_filter_size"] % 2 != 1
        or params["background_blur"] < 0
        or not 0 <= params["black_point"] < params["white_point"] <= 255
        or params["tone_gamma"] <= 0
        or (params["binary_threshold"] is not None and not 0 <= params["binary_threshold"] <= 255)):
        raise ValueError("Tham số tiền xử lý nằm ngoài phạm vi hợp lệ.")
    rgb = image.convert("RGB")
    hue, saturation, value = rgb.convert("HSV").split()
    red, green, _ = rgb.split()
    hue_mask = hue.point([255 if h <= params["red_hue_width"] or h >= 256-params["red_hue_width"] else 0
                          for h in range(256)])
    saturation_mask = saturation.point([255 if s >= params["red_min_saturation"] else 0 for s in range(256)])
    dominance = ImageChops.subtract(red, green).point([
        255 if d >= params["red_min_r_minus_g"] else 0 for d in range(256)])
    protected = value.point([255 if v >= params["red_protect_max_below"] else 0 for v in range(256)])
    red_mask = ImageChops.multiply(ImageChops.multiply(hue_mask, saturation_mask),
                                   ImageChops.multiply(dominance, protected))
    gray_original = ImageOps.grayscale(rgb)
    # Nền ước lượng bằng cửa sổ rộng hơn nét chữ; không blur ảnh chữ đưa vào OCR.
    background = gray_original.copy()
    background.thumbnail((params["background_max_side"], params["background_max_side"]))
    background = background.filter(ImageFilter.MaxFilter(params["background_filter_size"]))
    background = background.filter(ImageFilter.GaussianBlur(params["background_blur"]))
    background = background.resize(rgb.size, Image.Resampling.BILINEAR)
    no_red = Image.composite(background, gray_original, red_mask)
    # Chia theo từng dải để không giữ nhiều tensor/ảnh float toàn trang trong RAM.
    normalized = Image.new("L", rgb.size)
    for top in range(0, rgb.height, 256):
        box = (0, top, rgb.width, min(top+256, rgb.height))
        foreground = no_red.crop(box).convert("F")
        base = background.crop(box).point([max(1, v) for v in range(256)]).convert("F")
        stripe = ImageMath.lambda_eval(lambda a: a["fg"] * 255 / a["bg"], fg=foreground, bg=base).convert("L")
        normalized.paste(stripe, (0, top))
    black, white = params["black_point"], params["white_point"]
    curve = [round(255 * min(1.0, max(0.0, (v-black)/(white-black))) ** params["tone_gamma"])
             for v in range(256)]
    soft = normalized.point(curve)
    histogram = soft.histogram()
    threshold = params["binary_threshold"]
    if threshold is None:
        # Otsu trên ảnh đã cân nền/tone; chỉ là phương án so sánh, không chọn tự động.
        total = sum(histogram)
        sum_all = sum(i*count for i, count in enumerate(histogram))
        count_low, sum_low, best = 0, 0, -1.0
        threshold = 127
        for i, count in enumerate(histogram):
            count_low += count
            sum_low += i*count
            count_high = total-count_low
            if not count_low or not count_high:
                continue
            difference = sum_low/count_low - (sum_all-sum_low)/count_high
            score = count_low * count_high * difference * difference
            if score > best:
                best, threshold = score, i
    binary = soft.point([0 if v <= threshold else 255 for v in range(256)])
    overlay = Image.composite(Image.new("RGB", rgb.size, (255, 0, 180)), rgb, red_mask)
    mask_count = red_mask.histogram()[255]
    stats = {"algorithm": "bac_su_color_background_tone_v1", "source_size": list(rgb.size),
        "red_mask_pixels": mask_count, "red_mask_fraction": mask_count/(rgb.width*rgb.height),
        "white_fraction_soft": soft.histogram()[255]/(rgb.width*rgb.height),
        "binary_threshold": threshold, "output_mode": "RGB", "params": dict(params)}
    # Tất cả đầu vào OCR có ba kênh RGB, các kênh bằng nhau ở bản xám/nhị phân.
    return {"original": rgb, "red_mask": red_mask, "red_overlay": overlay,
        "gray_only": gray_original.convert("RGB"), "soft_gray": soft.convert("RGB"),
        "binary_otsu": binary.convert("RGB")}, stats




In [ ]:
def bac_main_preprocess_config():
    if BAC_PREPROCESS_MODE not in {"gray_scale", "binary"}:
        raise ValueError("BAC_PREPROCESS_MODE chỉ nhận gray_scale hoặc binary.")
    return {"algorithm": "bac_su_color_background_tone_v1", "mode": BAC_PREPROCESS_MODE,
            "params": dict(BAC_MAIN_PREPROCESS_PARAMS)}


def bac_main_ocr_input(source):
    """Cache PNG thực đưa vào Paddle; kiểm tra nguồn/cấu hình/hash trước khi dùng lại."""
    from PIL import Image
    config = bac_main_preprocess_config()
    source_sha = file_hash(source)
    identity = {"source_sha256": source_sha, "config": config}
    key = bytes_hash(json_bytes(identity))
    folder = LOCAL_CACHE / "bac_su_main_preprocess" / key
    path, meta_path = folder / "input.png", folder / "input.meta.json"
    if meta_path.is_file():
        record = json.loads(meta_path.read_text(encoding="utf-8"))
        if record.get("identity") != identity:
            raise FileExistsError("Cache tiền xử lý có cấu hình không khớp: " + str(meta_path))
        if path.is_file():
            if file_hash(path) != record.get("input_sha256"):
                raise FileExistsError("Ảnh cache tiền xử lý đã sửa ngoài notebook, giữ nguyên: " + str(path))
            if record.get("status") != "done":
                record["status"] = "done"
                atomic_write(meta_path, json_bytes(record))
            return path, record
    elif path.exists():
        raise FileExistsError("Ảnh cache thiếu metadata, giữ nguyên: " + str(path))
    with Image.open(source) as image:
        variants, stats = bac_su_preprocess(image, config["params"])
    variant = "soft_gray" if config["mode"] == "gray_scale" else "binary_otsu"
    chosen = variants[variant]
    buffer = io.BytesIO()
    chosen.save(buffer, format="PNG", compress_level=1)
    png = buffer.getvalue()
    record = {"status": "writing", "identity": identity, "mode": config["mode"],
        "variant": variant, "stats": stats, "input_sha256": bytes_hash(png),
        "pixel_sha256": bytes_hash(chosen.tobytes()), "size": list(chosen.size), "image_mode": chosen.mode}
    atomic_write(meta_path, json_bytes(record))
    atomic_write(path, png)
    record["status"] = "done"
    atomic_write(meta_path, json_bytes(record))
    return path, record


In [ ]:
def recognition_config(item, *, apply_bac_preprocess=True):
    if item["mode"] == "text":
        return {"mode": "copy_source_bytes"}
    common = {"engine": INFERENCE_ENGINE, "det_model": DETECTION_MODEL,
              "detection_max_side": DETECTION_MAX_SIDE, "device": DEVICE,
              "orientation": False, "unwarping": False,
              "versions": {k: v for k, v in runtime_versions.items() if k not in {"vietocr", "einops"}}}
    if JOBS[item["job"]]["kind"] == "han":
        policy = han_policy(item)
        horizontal = policy["layout"] == "horizontal_ltr"
        return {**common, "pipeline_revision": "bac_main_layout_v2" if item["job"] == "bac_su_han" else "source_layout_v1",
                **({"preprocessing": bac_main_preprocess_config() if apply_bac_preprocess else {"enabled": False}}
                   if item["job"] == "bac_su_han" else {}), "layout_policy": policy,
                "pipeline": "paddle_han_horizontal_rows" if horizontal else "paddle_han",
                "rec_model": RECOGNITION_MODEL, "rec_batch_size": HAN_REC_BATCH_SIZE,
                "reading_order": policy["reading_order"], "score_threshold": 0.0 if horizontal else REC_SCORE_THRESHOLD,
                "review_confidence": REVIEW_CONFIDENCE}
    return {**common, "versions": runtime_versions, "pipeline": "paddle_detection_vietocr", "rec_model": VIETOCR_MODEL,
            "model_identity": viet_model_identity, "reading_order": VIET_READING_ORDER,
            "crop_padding": VIET_CROP_PADDING, "max_width": VIETOCR_MAX_WIDTH,
            "max_length": VIETOCR_MAX_LENGTH, "review_confidence": VIET_REVIEW_CONFIDENCE,
            "box_threshold": DETECTION_BOX_THRESHOLD, "unclip_ratio": DETECTION_UNCLIP_RATIO}


def expected_identity(item, source_sha):
    return {"source_id": item["id"], "source_sha256": source_sha,
            "config_sha256": bytes_hash(json_bytes(recognition_config(item)))}

def check_saved(paths, identity):
    present = any(p.exists() for p in paths.values())
    if not present:
        return "new", None
    if not paths["meta"].is_file():
        raise FileExistsError(f"Có kết quả không có thông tin truy vết, giữ nguyên: {paths['txt']}")
    meta = json.loads(paths["meta"].read_bytes().decode("utf-8"))
    if meta.get("identity") != identity:
        raise FileExistsError(f"Nguồn/tham số thay đổi: {paths['txt']}. Đổi RUN_NAME để lưu phiên bản mới.")
    for key in paths:
        if key != "meta" and paths[key].exists():
            if file_hash(paths[key]) != meta.get("output_hashes", {}).get(key):
                raise FileExistsError(f"File đã được sửa ngoài notebook, giữ nguyên: {paths[key]}")
    complete = all(p.is_file() for key, p in paths.items() if key != "meta")
    if meta.get("status") == "done" and complete:
        return "done", meta
    if meta.get("status") == "writing":
        return "resume", meta
    raise FileExistsError(f"Kết quả thiếu file hoặc metadata không hợp lệ: {paths['txt']}")

def save_outputs(paths, outputs, metadata):
    metadata = {**metadata, "status": "writing",
                "output_hashes": {key: bytes_hash(data) for key, data in outputs.items()}}
    atomic_write(paths["meta"], json_bytes(metadata))
    for key, data in outputs.items():
        atomic_write(paths[key], data)
    metadata["status"] = "done"
    atomic_write(paths["meta"], json_bytes(metadata))
    return metadata

def text_order(data, mode, column_center_ratio=None):
    indices = list(range(len(data["rec_texts"])))
    if mode == "model" or not indices:
        return indices
    polygons = data.get("rec_polys")
    if polygons is None or len(polygons) != len(indices):
        raise ValueError("Không đủ tọa độ để xếp cột dọc.")
    if mode == "vertical_columns_rtl":
        return grouped_vertical_columns(data, BAC_COLUMN_CENTER_RATIO if column_center_ratio is None else column_center_ratio)["order"]
    if mode == "top_to_bottom":
        return sorted(indices, key=lambda i: (min(p[1] for p in polygons[i]), min(p[0] for p in polygons[i])))
    # Gợi ý hình học cho cột dọc: phải → trái, rồi trên → dưới.
    # Không đảm bảo đúng với trang ghép, nhiều khối hoặc dòng bị tách vụn.
    return sorted(indices, key=lambda i: (-sum(p[0] for p in polygons[i])/len(polygons[i]),
                                         min(p[1] for p in polygons[i])))

def predict_image(path, item, *, apply_bac_preprocess=True):
    is_han = JOBS[item["job"]]["kind"] == "han"
    policy = han_policy(item) if is_han else None
    ocr_path, preprocessing = Path(path), None
    if item["job"] == "bac_su_han" and apply_bac_preprocess:
        ocr_path, preprocessing = bac_main_ocr_input(path)
    if is_han and policy["layout"] == "horizontal_ltr":
        raw = predict_horizontal_han(ocr_path, policy)
    elif is_han:
        if ocr is None:
            raise RuntimeError("Chưa nạp PaddleOCR Hán ở mục 8.")
        predictions = list(ocr.predict(input=str(ocr_path), text_rec_score_thresh=REC_SCORE_THRESHOLD,
            text_det_limit_side_len=DETECTION_MAX_SIDE, text_det_limit_type="max"))
        if len(predictions) != 1:
            raise ValueError("Cần một ảnh một kết quả OCR.")
        raw = predictions[0].json
    else:
        raw = predict_viet_image(path)
    if not isinstance(raw, dict):
        raise TypeError("Kết quả OCR phải là dictionary.")
    data = raw.get("res", raw)
    texts, scores = data.get("rec_texts", []), data.get("rec_scores", [])
    if len(texts) != len(scores):
        raise ValueError("Số đoạn chữ và điểm tin cậy không khớp.")
    if is_han and policy["order_indices"] is not None:
        indices = checked_order(policy["order_indices"], len(texts))
    else:
        indices = text_order(data, policy["reading_order"] if is_han else VIET_READING_ORDER,
                             policy.get("column_center_ratio") if is_han else None)
    polygons = data.get("rec_polys", [])
    reading_map = [{"txt_line": rank+1, "index_model": index,
                    "polygon": polygons[index] if len(polygons) == len(texts) else None}
                   for rank, index in enumerate(indices)]
    text = "\n".join(texts[i] for i in indices)
    threshold = REVIEW_CONFIDENCE if is_han else VIET_REVIEW_CONFIDENCE
    geometry_review = han_geometry_review(data, policy) if is_han else {"flags": []}
    if is_han and policy["reading_order"] == "vertical_columns_rtl":
        grouping = grouped_vertical_columns(data, policy["column_center_ratio"])
        geometry_review["column_groups"] = grouping["groups"]
        geometry_review["ambiguous_column_indices"] = grouping["ambiguous_indices"]
        if grouping["ambiguous_indices"]: geometry_review["flags"].append("ambiguous_column_assignment")
    known_scores = [float(s) for s in scores if s is not None]
    return text, raw, {"pipeline": data.get("pipeline", "paddle_han" if is_han else "paddle_detection_vietocr"),
        "coordinate_space": "source_image_pixels", "reading_map": reading_map,
        **({"ocr_input_path": str(ocr_path), "preprocessing": preprocessing} if preprocessing else {}),
        "layout_flags": geometry_review["flags"], "layout_review": geometry_review,
        "text_count": len(texts), "order_indices": indices,
        "mean_confidence": sum(known_scores)/len(known_scores) if known_scores else None,
        "low_confidence_count": sum(s < threshold for s in known_scores),
        "unknown_confidence_count": len(scores) - len(known_scores),
        "line_warning_count": sum(bool(d["flags"]) for d in data.get("line_details", [])),
        "empty_prediction": not bool(text.strip())}


def process_item(item):
    started = time.perf_counter()
    source = input_path(item)
    identity = expected_identity(item, file_hash(source))
    paths = output_paths(item)
    state, metadata = check_saved(paths, identity)
    if state == "done":
        return {"status": "skipped", "txt": str(paths["txt"]), **metadata.get("review", {})}
    if item["mode"] == "text":
        outputs = {"txt": source.read_bytes()}
        review = {"copied_source_text": True, "empty_prediction": False}
    else:
        text, raw, review = predict_image(source, item)
        outputs = {"txt": text.encode("utf-8"), "json": json_bytes(raw)}
        if "image" in paths:
            outputs["image"] = Path(review["ocr_input_path"]).read_bytes()
            review["ocr_input_file"] = paths["image"].name
    metadata = {"identity": identity, "source": item, "config": recognition_config(item),
                "review": review, "saved_at_utc": datetime.now(timezone.utc).isoformat()}
    save_outputs(paths, outputs, metadata)
    return {"status": "copied" if item["mode"] == "text" else "ocr_done",
            "txt": str(paths["txt"]), "seconds": round(time.perf_counter()-started, 3), **review}

## 10. Xem ảnh mẫu và vùng chữ

- 10a: detection Việt, tùy chọn.
- 10b: mặc định chữ Hán, xem khung và crop của Paddle.
- 10c: kiểm tra crop/input/recognition Việt.

Các cell xem mẫu chưa ghi kết quả OCR hàng loạt. Thứ tự model/cột dọc vẫn cần đối chiếu ảnh nguồn.


### 10a. Xem detection một ảnh Việt — tùy chọn

Chỉ vẽ khung, chưa gọi VietOCR và không lưu sản phẩm. Chỉnh TEST_VIET_JOB/TEST_VIET_INDEX.


Số trên khung là thứ tự dòng TXT dự kiến (từ 1); bảng giữ ID detector gốc và tọa độ.
Các cell diagnostic 10c/10d vẫn chọn vùng bằng ID gốc, không dùng số dòng TXT.


In [ ]:
TEST_VIET_JOB = next((key for key in SELECTED_JOBS
    if JOBS[key]["mode"] == "ocr" and JOBS[key]["kind"] != "han"), None)
TEST_VIET_INDEX = 0
if TEST_VIET_JOB is None:
    print("Chưa chọn job ảnh Việt; bỏ qua detection mẫu.")
else:
    samples = [i for i in selected_items if i["job"] == TEST_VIET_JOB]
    if not 0 <= TEST_VIET_INDEX < len(samples):
        raise IndexError("TEST_VIET_INDEX nằm ngoài danh sách ảnh.")
    sample = samples[TEST_VIET_INDEX]
    path = input_path(sample)
    raw_detection, detection = detect_viet_regions(path)
    print(sample["relative_path"], "| số vùng:", len(detection["dt_polys"]))
    polygons = detection["dt_polys"]
    indices = text_order({"rec_texts": [""]*len(polygons), "rec_polys": polygons}, VIET_READING_ORDER)
    print("Số xanh = dòng TXT dự kiến (từ 1); index_model là ID detector gốc.")
    display_detection_image(path, polygons, order_indices=indices)
    display(pd.DataFrame([{"dong_txt": rank+1, "index_model": index,
        "diem_detection": detection["dt_scores"][index], "toa_do_anh_nguon": polygons[index]}
        for rank, index in enumerate(indices)]))


### 10b. Xem ảnh Hán, thứ tự TXT và crop thực

Chọn `TEST_JOB="ly_tran_han"`, `"thi_dinh_han"` hoặc `"bac_su_han"`, rồi đổi `TEST_INDEX`.
Số xanh trên ảnh là **dòng TXT từ 1**. Bảng được xếp cùng thứ tự TXT, giữ `index_model`
từ 0 để truy lại JSON và chọn `PADDLE_CROP_IDS`. Cột tọa độ luôn theo pixel ảnh nguồn.

Lý–Trần: crop nguyên hàng ngang trên ảnh nguồn, gồm cả dấu câu cuối hàng; preview giữ đúng pixel crop.
Thi Đình/Bắc sứ: dùng `CropByPolys` của Paddle, có thể nắn/xoay cột trước recognition.
Các ảnh hiển thị là JPEG thu nhỏ, không sửa ảnh gốc hoặc tensor model.

Nếu hàng còn gộp/tách sai, chỉnh `HAN_IMAGE_OVERRIDES` theo đúng `job/relative_path`.
`row_settings` đổi ngưỡng cho riêng ảnh; `row_boxes` khai báo các hình chữ nhật nguồn khi cần cắt tay.
`order_indices` chỉ nhận hoán vị đầy đủ các ID vùng, không dùng để bỏ dấu câu hoặc xóa vùng.
Sau khi đổi cách cắt/đọc: dùng `RUN_NAME` mới, chạy lại cấu hình, mục 8–9 rồi xem lại 10b.


In [ ]:
def paddle_han_crop_preview(path, polygons, indices):
    """Tái tạo crop trước recognition bằng đúng CropByPolys của PaddleX 3.7."""
    import math
    from PIL import Image

    indices = list(indices)
    if len(set(indices)) != len(indices) or any(not isinstance(i, int) or not 0 <= i < len(polygons) for i in indices):
        raise ValueError("ID crop phải là index_model hợp lệ và không trùng.")
    if not indices:
        return []
    import numpy as np
    from paddlex.inference.pipelines.components.common.crop_image_regions import CropByPolys

    with Image.open(path) as original:
        source = original.convert("RGB")
    # Phép crop hình học độc lập thứ tự kênh màu; dùng RGB để hiển thị PIL đúng màu.
    # Cùng toán tử nắn/xoay như pipeline Hán, chưa chạy resize/tensor recognition.
    crops = CropByPolys(det_box_type="quad")(np.asarray(source), [polygons[i] for i in indices])
    if len(crops) != len(indices):
        raise ValueError("Số crop Paddle không khớp các vùng đã chọn.")
    output = []
    for index, array in zip(indices, crops):
        image = Image.fromarray(array)
        points = polygons[index]
        distance = lambda a, b: math.hypot(a[0]-b[0], a[1]-b[1])
        source_w = max(distance(points[0], points[1]), distance(points[2], points[3]))
        source_h = max(distance(points[0], points[3]), distance(points[1], points[2]))
        output.append({"index_model": index, "source_w": round(source_w, 1),
            "source_h": round(source_h, 1), "source_h_w": round(source_h/max(source_w, 1), 3),
            "crop_w": image.width, "crop_h": image.height,
            "crop_h_w": round(image.height/max(image.width, 1), 3), "image": image})
    return output


def han_crop_preview(path, data, indices):
    """Preview đúng pixel crop nhận dạng; Hán dọc giữ toán tử Paddle hiện có."""
    if data.get("pipeline") != "paddle_han_horizontal_rows":
        return paddle_han_crop_preview(path, data.get("rec_polys", []), indices)
    from PIL import Image
    indices = list(indices)
    details = data["line_details"]
    if len(set(indices)) != len(indices) or any(type(i) is not int or not 0 <= i < len(details) for i in indices):
        raise ValueError("ID hàng ngang không hợp lệ hoặc trùng.")
    with Image.open(path) as original:
        source = original.convert("RGB")
    rows = []
    for index in indices:
        image = source.crop(details[index]["crop_box"])
        rows.append({"index_model": index, "source_w": image.width, "source_h": image.height,
            "source_h_w": image.height/max(image.width, 1), "crop_w": image.width,
            "crop_h": image.height, "crop_h_w": image.height/max(image.width, 1), "image": image})
    return rows


In [ ]:
from PIL import Image
from IPython.display import Image as NotebookImage

# ===== Mẫu Hán: đổi job hoặc chỉ số ảnh tại đây =====
TEST_JOB = next((key for key in SELECTED_JOBS
    if JOBS[key]["mode"] == "ocr" and JOBS[key]["kind"] == "han"), None)
# Ví dụ: TEST_JOB = "thi_dinh_han", "ly_tran_han" hoặc "bac_su_han".
TEST_INDEX = 0
SHOW_PADDLE_CROPS = True
PADDLE_CROP_IDS = None             # None: vài vùng đầu theo thứ tự TXT; hoặc [0, 1, 2].
PADDLE_CROP_COUNT = 3

if TEST_JOB is None:
    print("Chưa chọn job Hán. Thêm thi_dinh_han/ly_tran_han/bac_su_han vào SELECTED_JOBS, chạy lại kiểm kê và nạp model.")
else:
    if TEST_JOB not in SELECTED_JOBS or JOBS[TEST_JOB]["kind"] != "han":
        raise ValueError("Mục 10b đang xem Hán; TEST_JOB phải là job Hán đã chọn.")
    samples = [i for i in selected_items if i["job"] == TEST_JOB and i["mode"] == "ocr"]
    if not 0 <= TEST_INDEX < len(samples):
        raise IndexError(f"TEST_INDEX phải từ 0 đến {len(samples)-1}.")
    sample = samples[TEST_INDEX]
    path = input_path(sample)
    sample_text, raw_prediction, review = predict_image(path, sample)
    data = raw_prediction.get("res", raw_prediction)
    polygons = data.get("rec_polys", [])
    print("Nguồn Hán:", sample["relative_path"], "|", review["pipeline"])
    print("Số xanh = dòng TXT (từ 1). ID crop dùng cột index_model (từ 0) trong bảng.")
    preview_path = Path(review.get("ocr_input_path", path))
    print("Tiền xử lý:", review.get("preprocessing", {}).get("mode", "không áp dụng"))
    display_detection_image(preview_path, polygons, order_indices=review["order_indices"])
    rank = {index: order for order, index in enumerate(review["order_indices"])}
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame([{"dong_txt": rank[n]+1, "index_model": n, "chu": data["rec_texts"][n],
            "diem_recognition": data["rec_scores"][n], "toa_do_anh_nguon": polygons[n]}
            for n in review["order_indices"]]))
    print("TXT theo thứ tự đã chọn:")
    print(sample_text)
    if SHOW_PADDLE_CROPS:
        if not isinstance(PADDLE_CROP_COUNT, int) or PADDLE_CROP_COUNT < 1:
            raise ValueError("PADDLE_CROP_COUNT phải là số nguyên dương.")
        crop_ids = review["order_indices"][:PADDLE_CROP_COUNT] if PADDLE_CROP_IDS is None else PADDLE_CROP_IDS
        han_crops = han_crop_preview(preview_path, data, crop_ids)
        display(pd.DataFrame([{k: v for k, v in row.items() if k != "image"} for row in han_crops]))
        for row in han_crops:
            index = row["index_model"]
            print("Dòng TXT", rank[index]+1, "| index_model", index, "| nguồn W/H:", row["source_w"], row["source_h"],
                  "| crop W/H:", row["crop_w"], row["crop_h"])
            print("Recognition:", data["rec_texts"][index])
            thumbnail = row["image"].copy()
            thumbnail.thumbnail((1400, 400))
            buffer = io.BytesIO()
            thumbnail.save(buffer, format="JPEG", quality=95)
            display(NotebookImage(data=buffer.getvalue(), format="jpeg"))
        print("Crop trước resize recognition: Lý–Trần giữ hàng ngang; Paddle có thể xoay cột dọc. Chưa phải tensor input.")
    print("Cảnh báo bố cục:", review["layout_flags"])
    print("Vùng cần rà soát:", review["layout_review"])


### 10c. Kiểm tra crop → input thực → recognition

Chạy sau mục 8–9. Cùng một tensor được hiển thị rồi đưa vào decoder, không tạo ảnh resize giả lập.
So sánh chiều rộng bằng DEBUG_WIDTHS, giới hạn giải mã bằng DEBUG_DECODE_LIMITS; dùng lại model đang nạp.
ID dòng là số xanh trên ảnh detection, không phải số dòng đọc từ trên xuống.
Nhập DEBUG_REFERENCES để đo CER thô; bỏ trống thì độ chính xác chưa được đánh giá.
Không nối câu, sửa chữ hay thay đổi tham số OCR hàng loạt. DEBUG_SAVE=False mặc định không ghi file.
Nếu bật lưu, ảnh JPEG chỉ là bản xem; tensor/shape và dự đoán gốc được ghi riêng trong 05_quan_ly.


In [ ]:
def prepared_input_image(batch):
    """Xem tensor NCHW [0,1] thực sự đưa vào VietOCR; không resize lại để giả lập."""
    import math
    from PIL import Image

    shape = tuple(batch.shape)
    if len(shape) != 4 or shape[0] != 1 or shape[1] != 3 or min(shape[2:]) < 1:
        raise ValueError(f"Cần tensor (1, 3, H, W), nhận {shape}.")
    channels = batch.detach().cpu().tolist()[0]
    pixels, minimum, maximum = [], float("inf"), float("-inf")
    for y in range(shape[2]):
        for x in range(shape[3]):
            values = [float(channels[c][y][x]) for c in range(3)]
            if not all(math.isfinite(v) and 0 <= v <= 1 for v in values):
                raise ValueError("Input không phải RGB hữu hạn trong [0,1]; kiểm tra preprocessing trước khi hiển thị.")
            minimum, maximum = min(minimum, *values), max(maximum, *values)
            pixels.append(tuple(round(v * 255) for v in values))
    image = Image.new("RGB", (shape[3], shape[2]))
    image.putdata(pixels)
    return image, {"input_shape": list(shape), "input_dtype": str(batch.dtype),
                   "input_min": minimum, "input_max": maximum,
                   "input_rgb_sha256": bytes_hash(image.tobytes())}


def character_error_report(reference, prediction):
    """CER thô theo code point, giữ nguyên dấu/khoảng trắng; không sửa dự đoán."""
    if reference is None:
        return {"reference_provided": False, "cer": None, "edit_distance": None,
                "substitutions": None, "deletions": None, "insertions": None, "exact_match": None}
    if not isinstance(reference, str):
        raise TypeError("Dòng chuẩn phải là chuỗi hoặc None.")
    # Trạng thái: tổng lỗi, thay thế, xóa, chèn; tie ưu tiên đường chéo.
    previous = [(j, 0, 0, j) for j in range(len(prediction) + 1)]
    for i, expected in enumerate(reference, 1):
        current = [(i, 0, i, 0)]
        for j, actual in enumerate(prediction, 1):
            cost = int(expected != actual)
            d, s, deleted, added = previous[j-1]
            diagonal = (d + cost, s + cost, deleted, added)
            d, s, deleted, added = previous[j]
            deletion = (d+1, s, deleted+1, added)
            d, s, deleted, added = current[j-1]
            insertion = (d+1, s, deleted, added+1)
            current.append(min((diagonal, deletion, insertion), key=lambda state: state[0]))
        previous = current
    distance, substitutions, deletions, insertions = previous[-1]
    return {"reference_provided": True, "cer": distance/len(reference) if reference else None,
            "edit_distance": distance, "substitutions": substitutions, "deletions": deletions,
            "insertions": insertions, "exact_match": reference == prediction}


def diagnose_viet_image(path, line_ids=None, max_lines=3, widths=None, decode_limits=None, references=None):
    import math
    from PIL import Image

    if not all(hasattr(viet_recognizer, name) for name in ("prepare_input", "predict_prepared")):
        raise RuntimeError("Chạy lại helper VietOCR và cell nạp model mục 8 để có API diagnostic mới.")
    widths = list(dict.fromkeys(widths if widths is not None else [VIETOCR_MAX_WIDTH]))
    decode_limits = list(dict.fromkeys(decode_limits if decode_limits is not None else [VIETOCR_MAX_LENGTH]))
    if not widths or any(not isinstance(w, int) or w < 32 for w in widths):
        raise ValueError("Danh sách widths cần các số nguyên >=32.")
    if not decode_limits or any(not isinstance(n, int) or n <= 0 for n in decode_limits):
        raise ValueError("Danh sách decode_limits cần số nguyên dương.")
    if not isinstance(max_lines, int) or max_lines < 1:
        raise ValueError("max_lines phải là số nguyên dương.")
    references = {} if references is None else references
    if not isinstance(references, dict) or any(not isinstance(k, int) for k in references):
        raise ValueError("references phải là dictionary dùng ID detection dạng số nguyên.")
    raw_detection, detection = detect_viet_regions(path)
    polygons = detection["dt_polys"]
    if any(not 0 <= k < len(polygons) for k in references):
        raise ValueError("ID dòng chuẩn nằm ngoài kết quả detection.")
    order = text_order({"rec_texts": [""]*len(polygons), "rec_polys": polygons}, VIET_READING_ORDER)
    if line_ids is None:
        line_ids = order[:max_lines]
    else:
        line_ids = list(line_ids)
        if len(set(line_ids)) != len(line_ids) or any(not isinstance(i, int) or not 0 <= i < len(polygons) for i in line_ids):
            raise ValueError(f"line_ids cần ID detection không trùng, từ 0 đến {len(polygons)-1}.")
    with Image.open(path) as original:
        image = original.convert("RGB")
    artifacts, rows = {}, []
    rank = {index: position for position, index in enumerate(order)}
    for index in line_ids:
        crop, crop_quad = line_crop(image, polygons[index], VIET_CROP_PADDING)
        artifacts[index] = {"crop": crop, "inputs": {}}
        for width in widths:
            prepared = viet_recognizer.prepare_input(crop, max_width=width)
            input_image, statistics = prepared_input_image(prepared)
            artifacts[index]["inputs"][width] = input_image
            for limit in decode_limits:
                text, score, trace = viet_recognizer.predict_prepared(prepared, max_length=limit, return_trace=True)
                score = float(score) if score is not None and math.isfinite(float(score)) else None
                rows.append({"line_id": index, "reading_rank": rank[index],
                    "source_path": str(path), "crop_size": list(crop.size), "crop_quad": crop_quad,
                    "width_cap": width, "width_clamped": input_image.width < math.ceil(
                        (int(input_image.height * crop.width / crop.height))/10)*10,
                    **statistics, **trace, "confidence": score, "prediction": text,
                    "reference": references.get(index), **character_error_report(references.get(index), text)})
    return rows, artifacts, raw_detection


def display_diagnostic_line(image, title, zoom=1):
    from IPython.display import Image as NotebookImage, display

    print(title, "| kích thước pixel thực:", image.size)
    preview = image.copy()
    # Chỉ thu nhỏ bản xem; tensor dùng để decode giữ nguyên.
    if zoom == 1:
        preview.thumbnail((1500, 400))
    buffer = io.BytesIO()
    preview.save(buffer, format="JPEG", quality=95)
    display(NotebookImage(data=buffer.getvalue(), format="jpeg",
                          width=preview.width*zoom, height=preview.height*zoom))


In [ ]:
# ===== Chỉ chỉnh cấu hình diagnostic ở cell này =====
DEBUG_RUN = True
DEBUG_JOB = next((key for key in SELECTED_JOBS
    if JOBS[key]["mode"] == "ocr" and JOBS[key]["kind"] != "han"), None)
DEBUG_IMAGE_INDEX = 0
DEBUG_LINE_IDS = None                 # None: lấy các dòng đầu theo thứ tự đọc; hoặc [17, 16].
DEBUG_MAX_LINES = 3
DEBUG_WIDTHS = [VIETOCR_MAX_WIDTH]       # Mặc định 512; có thể đổi thành [512, 1024] để so sánh.
DEBUG_DECODE_LIMITS = [VIETOCR_MAX_LENGTH]      # Có thể thử [128, 256].
DEBUG_REFERENCES = {}                 # Ví dụ {17: "Gõ NGUYÊN dòng chuẩn trong ảnh"}.
DEBUG_SAVE = False                    # True: xuất ảnh crop/input + CSV/JSON vào 05_quan_ly.

if not DEBUG_RUN or DEBUG_JOB is None:
    print("Bỏ qua diagnostic: chưa bật hoặc không chọn ảnh Việt.")
else:
    if DEBUG_JOB not in SELECTED_JOBS or JOBS[DEBUG_JOB]["mode"] != "ocr" or JOBS[DEBUG_JOB]["kind"] == "han":
        raise ValueError("DEBUG_JOB phải là một job ảnh Việt trong SELECTED_JOBS.")
    debug_samples = [i for i in selected_items if i["job"] == DEBUG_JOB]
    if not 0 <= DEBUG_IMAGE_INDEX < len(debug_samples):
        raise IndexError("DEBUG_IMAGE_INDEX nằm ngoài danh sách kiểm kê.")
    debug_sample = debug_samples[DEBUG_IMAGE_INDEX]
    debug_path = input_path(debug_sample)
    debug_rows, debug_artifacts, debug_detection = diagnose_viet_image(
        debug_path, DEBUG_LINE_IDS, DEBUG_MAX_LINES, DEBUG_WIDTHS,
        DEBUG_DECODE_LIMITS, DEBUG_REFERENCES)
    print("Nguồn:", debug_sample["relative_path"])
    display_detection_image(debug_path, debug_detection.get("res", debug_detection)["dt_polys"])
    for line_id, artifact in debug_artifacts.items():
        print("\n===== ID detection", line_id, "=====")
        display_diagnostic_line(artifact["crop"], "A. Ảnh dòng sau crop/nắn")
        for width, model_image in artifact["inputs"].items():
            display_diagnostic_line(model_image, f"B. Ảnh khôi phục từ tensor input, width_cap={width}", zoom=2)
            for row in [r for r in debug_rows if r["line_id"] == line_id and r["width_cap"] == width]:
                print("C. Recognition:", row["prediction"])
                print("shape:", row["input_shape"], "| dừng:", row["stop_reason"],
                      "| token đã sinh:", row["generated_tokens"], "| giới hạn:", row["decode_limit"],
                      "| confidence:", row["confidence"])
                if row["reference_provided"]:
                    print("Tham chiếu:", row["reference"], "| CER:", row["cer"],
                          "| thay/xóa/chèn:", row["substitutions"], row["deletions"], row["insertions"])
                else:
                    print("Chưa có dòng chuẩn: chưa đo được độ chính xác/đủ nội dung.")
    debug_table = pd.DataFrame(debug_rows)
    if not debug_table.empty:
        columns = ["line_id", "reading_rank", "width_cap", "input_shape", "width_clamped",
                   "decode_limit", "generated_tokens", "stop_reason", "confidence", "prediction", "cer", "deletions"]
        with pd.option_context("display.max_colwidth", None):
            display(debug_table[columns])
    if DEBUG_SAVE:
        debug_dir = report_dir / "kiem_tra_vietocr" / (
            datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8])
        debug_dir.mkdir(parents=True, exist_ok=False)
        atomic_write(debug_dir / "recognition.json", json_bytes({
            "source": debug_sample, "source_sha256": file_hash(debug_path),
            "config": recognition_config(debug_sample), "rows": debug_rows, "detector": debug_detection}))
        atomic_write(debug_dir / "recognition.csv", debug_table.to_csv(index=False).encode("utf-8-sig"))
        for line_id, artifact in debug_artifacts.items():
            artifact["crop"].save(debug_dir / f"line_{line_id:03d}_crop.png", compress_level=0)
            for width, model_image in artifact["inputs"].items():
                model_image.save(debug_dir / f"line_{line_id:03d}_input_cap{width}.png", compress_level=0)
        print("Đã lưu diagnostic riêng:", debug_dir)
    print("EOS là lý do dừng, chưa chứng minh đọc đủ. Confidence không phải độ chính xác.")


### 10d. So sánh recognition Hán trên cùng crop (tùy chọn)

Batch size 1 giữ theo phép thử của bạn. Mục này so `PP-OCRv6_medium_rec` và
`PP-OCRv5_server_rec`, dùng cùng detector/crop và batch 1 cho cả hai.
V5 là ứng viên có hỗ trợ chữ phồn thể, chữ dọc và viết tay; chưa kết luận tốt hơn trên sách này.

Chạy helper mục 10c (định nghĩa `character_error_report`; không cần nạp VietOCR nếu chỉ chọn Hán),
rồi hai cell dưới. Bật `HAN_COMPARE_RUN=True`, chọn job/ảnh/ID đã xem ở 10b.
Để `{}` khi chưa có bản chuẩn; muốn đo CER phải nhập đủ chữ của toàn vùng, giữ nguyên dị thể.
Nên kiểm tra nhiều cột ngắn/dài của cả Lý–Trần và Thi Đình trước khi đổi model chính.

Hai model được nạp lần lượt rồi đóng. Cell giữ nguyên `ocr`, `RECOGNITION_MODEL` và các TXT đã có.
Không cài lại Torch/Paddle GPU. Báo cáo tùy chọn nằm riêng trong `05_quan_ly/so_sanh_recognition/`.
Nếu chọn model mới để chạy chính thức: đổi `RECOGNITION_MODEL`, `RUN_NAME`, rồi chạy lại mục 6–9 và thử 10b.
Không chọn model chỉ vì confidence cao; CER cần người kiểm tra Hán văn đối chiếu ảnh gốc.

Nguồn: [Paddle recognition](https://www.paddleocr.ai/latest/en/version3.x/module_usage/text_recognition.html),
[checkpoint v5 chính thức](https://huggingface.co/PaddlePaddle/PP-OCRv5_server_rec_safetensors).


In [ ]:
def compare_han_recognition(crops, model_names, references=None, recognizer_factory=None):
    """Cùng crop PNG, mỗi lần một vùng; không đổi model/cấu hình OCR hàng loạt."""
    import tempfile

    references = {} if references is None else references
    crops, model_names = list(crops), list(model_names)
    ids = [c["index_model"] for c in crops]
    if not crops or len(set(ids)) != len(ids):
        raise ValueError("Chọn ít nhất một crop, các ID không trùng.")
    if not model_names or len(set(model_names)) != len(model_names):
        raise ValueError("Chọn model, tên model không trùng.")
    if set(references) - set(ids):
        raise ValueError("Dòng chuẩn có ID nằm ngoài các crop đang so sánh.")
    if any(not isinstance(text, str) or not text for text in references.values()):
        raise ValueError("Dòng chuẩn phải là toàn bộ vùng chữ, không để chuỗi rỗng.")
    if recognizer_factory is None:
        from paddleocr import TextRecognition
        recognizer_factory = lambda name: TextRecognition(
            model_name=name, engine=INFERENCE_ENGINE, device=DEVICE)

    # Dùng đường dẫn PNG để ReadImage của cả hai model tự xử lý màu như nhau.
    # Không truyền nhầm RGB ndarray vào API nhận ảnh BGR.
    LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
    rows = []
    with tempfile.TemporaryDirectory(prefix="han_rec_compare_", dir=LOCAL_CACHE) as scratch:
        inputs = []
        for crop in crops:
            image = crop["image"].convert("RGB")
            filename = Path(scratch) / f"crop_{crop['index_model']:04d}.png"
            image.save(filename, format="PNG", compress_level=0)
            inputs.append((crop["index_model"], filename, image.size, file_hash(filename)))
        for model_name in model_names:
            recognizer = recognizer_factory(model_name)
            try:
                for index, filename, size, crop_sha in inputs:
                    predictions = list(recognizer.predict(input=str(filename), batch_size=1))
                    if len(predictions) != 1:
                        raise ValueError("Một crop phải trả đúng một kết quả recognition.")
                    raw = predictions[0].json
                    raw = json.loads(raw) if isinstance(raw, str) else raw
                    result = raw.get("res", raw)
                    prediction = result["rec_text"]
                    if not isinstance(prediction, str):
                        raise TypeError("rec_text phải là chuỗi.")
                    reference = references.get(index)
                    rows.append({"model": model_name, "index_model": index,
                        "crop_w": size[0], "crop_h": size[1], "crop_sha256": crop_sha,
                        "batch_size": 1, "prediction": prediction,
                        "confidence": float(result["rec_score"]), "reference": reference,
                        **character_error_report(reference, prediction)})
            finally:
                # Chỉ đóng model thử nghiệm; không đóng instance ocr đang dùng.
                recognizer.close()
    summaries = []
    for model_name in model_names:
        measured = [r for r in rows if r["model"] == model_name and r["reference_provided"]]
        characters = sum(len(r["reference"]) for r in measured)
        errors = sum(r["edit_distance"] for r in measured)
        summaries.append({"model": model_name, "checked_regions": len(measured),
            "reference_characters": characters, "edit_distance": errors if measured else None,
            "cer": errors/characters if characters else None})
    return rows, summaries


In [ ]:
# ===== 10d: so recognition Hán trên crop cố định =====
HAN_COMPARE_RUN = False                # True khi muốn tải/chạy model ứng viên.
HAN_COMPARE_JOB = "thi_dinh_han"        # Hoặc ly_tran_han / bac_su_han.
HAN_COMPARE_IMAGE_INDEX = 0
HAN_COMPARE_IDS = None                 # Ví dụ [11, 6, 10, 12]; ID chỉ đúng với ảnh đã xem.
HAN_COMPARE_MAX_CROPS = 3
HAN_COMPARE_MODELS = ["PP-OCRv6_medium_rec", "PP-OCRv5_server_rec"]
HAN_COMPARE_REFERENCES = {}            # {ID: "toàn bộ chữ chuẩn của vùng ID"}; không nhập đoạn ngắn.
HAN_COMPARE_SAVE = False

if HAN_COMPARE_RUN:
    if HAN_COMPARE_JOB not in SELECTED_JOBS or JOBS[HAN_COMPARE_JOB]["kind"] != "han":
        raise ValueError("Chọn job Hán đã kiểm kê và nạp model ở mục 6–9.")
    candidates = [i for i in selected_items if i["job"] == HAN_COMPARE_JOB and i["mode"] == "ocr"]
    if not 0 <= HAN_COMPARE_IMAGE_INDEX < len(candidates):
        raise IndexError(f"Chỉ số ảnh phải từ 0 đến {len(candidates)-1}.")
    compare_sample = candidates[HAN_COMPARE_IMAGE_INDEX]
    compare_path = input_path(compare_sample)
    _, compare_raw, compare_review = predict_image(compare_path, compare_sample)
    compare_data = compare_raw.get("res", compare_raw)
    if not isinstance(HAN_COMPARE_MAX_CROPS, int) or HAN_COMPARE_MAX_CROPS < 1:
        raise ValueError("HAN_COMPARE_MAX_CROPS phải là số nguyên dương.")
    compare_ids = (compare_review["order_indices"][:HAN_COMPARE_MAX_CROPS]
                   if HAN_COMPARE_IDS is None else HAN_COMPARE_IDS)
    compare_crops = han_crop_preview(Path(compare_review.get("ocr_input_path", compare_path)), compare_data, compare_ids)
    han_compare_rows, han_compare_summary = compare_han_recognition(
        compare_crops, HAN_COMPARE_MODELS, HAN_COMPARE_REFERENCES)
    print("Nguồn:", compare_sample["relative_path"])
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(han_compare_rows).drop(columns=["crop_sha256"]))
        display(pd.DataFrame(han_compare_summary))
    print("CER trống khi chưa có dòng chuẩn. Confidence giữa model không dùng để chọn model tốt hơn.")
    if HAN_COMPARE_SAVE:
        compare_dir = RUN_ROOT / "05_quan_ly" / "so_sanh_recognition" / (
            datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
        compare_dir.mkdir(parents=True, exist_ok=False)
        report = {"source": compare_sample["relative_path"], "source_sha256": file_hash(compare_path),
            "engine": INFERENCE_ENGINE, "device": DEVICE, "versions": runtime_versions,
            "rows": han_compare_rows, "summary": han_compare_summary}
        (compare_dir / "comparison.json").write_bytes(json_bytes(report))
        pd.DataFrame(han_compare_rows).to_csv(compare_dir / "comparison.csv", index=False, encoding="utf-8-sig")
        print("Đã lưu phép thử riêng:", compare_dir)
else:
    print("10d đang tắt. Đặt HAN_COMPARE_RUN=True khi muốn so model Hán; không chạy hàng loạt.")


### 10e. Thử khử đỏ, nền vàng và chữ xuyên trang — chỉ Bắc sứ Hán

10e là thử nghiệm riêng. Luồng chính 10b/11 đã dùng cấu hình Bắc sứ ở cell 1.
Cần mục 1–7, helper lõi `bac_preprocess_core` ở mục 9 và helper thử bên dưới; xem ảnh không cần model/GPU.
Helper xử lý ảnh nằm ở mục 9; chạy helper thử bên dưới, bật `BAC_PREPROCESS_RUN=True` rồi chạy cell cấu hình 10e.
Mỗi lần chỉnh tham số, chạy lại cell cấu hình; không cần cài thư viện hoặc chạy OCR hàng loạt.

Ảnh RGB → mặt nạ đỏ HSV → thay đỏ bằng nền ước lượng → cân nền → xám/tone → thử Otsu.
Mặt nạ lấy từ RGB trước khi mất thông tin màu. Vùng rất tối được bảo vệ; mặt nạ không nở sang nét lân cận.
Các biến thể OCR có ba kênh RGB bằng nhau, giữ nguyên kích thước/tọa độ ảnh nguồn.
Không dùng ảnh thu nhỏ hiển thị để OCR; PNG xuất là ảnh đầy đủ độ phân giải.

`white_point` thấp xóa vùng mờ mạnh hơn nhưng có thể mất nét chính nhạt.
`red_protect_max_below` cao bảo vệ nhiều nét tối hơn nhưng có thể để lại đỏ sẫm.
Chỉ chọn sau khi xem cùng vùng phóng to, nhất là nét mảnh và giao điểm đỏ/đen.
Không bảo đảm xóa toàn bộ chữ xuyên trang khi độ đậm của nó trùng với chữ mặt trước.

`BAC_PREPROCESS_SAVE=True` lưu PNG/ảnh so sánh/tham số/hash vào
`05_quan_ly/thu_tien_xu_ly_bac_su/<timestamp_uuid>/`; ảnh nguồn và bộ OCR cũ giữ nguyên.
`BAC_PREPROCESS_OCR=True` cần nạp v6 + batch 1 ở mục 8–9; lưu OCR thử riêng cho ảnh gốc và ba biến thể.
Detection có thể thay đổi giữa các biến thể. Không so confidence để kết luận accuracy, không tự chọn bản thắng.


In [ ]:
def bac_su_comparison_board(variants, box=None, tile_size=(330, 650)):
    from PIL import Image, ImageDraw, ImageOps

    if box is not None:
        width, height = variants["original"].size
        if (len(box) != 4 or any(type(v) is not int for v in box)
            or not 0 <= box[0] < box[2] <= width or not 0 <= box[1] < box[3] <= height):
            raise ValueError("Vùng zoom phải nằm trong ảnh gốc, dùng pixel nguyên.")
    names = ["original", "red_overlay", "gray_only", "soft_gray", "binary_otsu"]
    w, h = tile_size
    board = Image.new("RGB", (len(names)*w, h+35), "#eeeeee")
    draw = ImageDraw.Draw(board)
    for index, name in enumerate(names):
        source = variants[name].crop(box) if box is not None else variants[name]
        tile = ImageOps.contain(source, (w-8, h-8))
        board.paste(tile, (index*w+(w-tile.width)//2, 35+(h-tile.height)//2))
        draw.text((index*w+5, 10), name, fill="black")
    return board


def save_bac_su_trial(root, sample, source_path, variants, stats, zoom_box=None):
    """Một thư mục mới cho mỗi phép thử; không gọi output_paths/process_item."""
    if sample["job"] != "bac_su_han" or sample["mode"] != "ocr":
        raise ValueError("Thử nghiệm chỉ áp dụng cho ảnh Bắc sứ Hán.")
    folder = Path(root) / "05_quan_ly" / "thu_tien_xu_ly_bac_su" / (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
    folder.mkdir(parents=True, exist_ok=False)
    board = bac_su_comparison_board(variants)
    board.save(folder / "comparison.jpg", quality=95)
    if zoom_box is not None:
        bac_su_comparison_board(variants, zoom_box, (400, 600)).save(folder / "zoom.jpg", quality=95)
    hashes = {}
    for name in ["red_mask", "gray_only", "soft_gray", "binary_otsu"]:
        filename = folder / (name + ".png")
        variants[name].save(filename, format="PNG", compress_level=1)
        hashes[name] = {"file_sha256": file_hash(filename),
                        "pixel_sha256": bytes_hash(variants[name].tobytes())}
    report = {"source": sample, "source_sha256": file_hash(source_path), "preprocessing": stats,
        "coordinate_space": "source_image_pixels", "zoom_box": zoom_box,
        "output_hashes": hashes, "applied_to_production": False, "ocr_results": {}}
    (folder / "trial.json").write_bytes(json_bytes(report))
    return folder, report


def compare_bac_su_trial_ocr(folder, report, sample, original_path):
    """So toàn pipeline từng biến thể: detection có thể khác; chưa đo accuracy/CER."""
    if sample["job"] != "bac_su_han" or globals().get("ocr") is None:
        raise RuntimeError("Chọn Bắc sứ Hán và nạp model ở mục 8 trước khi thử OCR.")
    if RECOGNITION_MODEL != "PP-OCRv6_medium_rec" or HAN_REC_BATCH_SIZE != 1:
        raise ValueError("Phép thử giữ recognition v6 và batch size 1.")
    rows = []
    for name in ["original", "gray_only", "soft_gray", "binary_otsu"]:
        path = Path(original_path) if name == "original" else Path(folder) / (name + ".png")
        text, raw, review = predict_image(path, sample, apply_bac_preprocess=False)
        (Path(folder) / (name + "_ocr.txt")).write_text(text, encoding="utf-8")
        (Path(folder) / (name + "_ocr.json")).write_bytes(json_bytes(raw))
        report["ocr_results"][name] = review
        rows.append({"variant": name, "regions": review["text_count"],
            "mean_confidence": review["mean_confidence"], "characters": len(text),
            "accuracy_measured": False})
    (Path(folder) / "trial.json").write_bytes(json_bytes(report))
    return rows


In [ ]:
# ===== 10e: thử tiền xử lý Bắc sứ Hán; không đổi OCR hàng loạt =====
BAC_PREPROCESS_RUN = False              # True để xem ảnh thử.
BAC_PREPROCESS_INDICES = [0]            # Có thể thử [0, 12, 29]; theo kiểm kê hiện tại.
BAC_PREPROCESS_ZOOM_BOX = None          # [x1, y1, x2, y2] pixel nguồn; chỉ dùng khi chọn một ảnh.
BAC_PREPROCESS_SAVE = True              # PNG đầy đủ độ phân giải + báo cáo trong thư mục riêng.
BAC_PREPROCESS_OCR = False              # True: so cả pipeline trên 4 biến thể bằng v6/batch 1.
BAC_PREPROCESS_PARAMS = {
    "red_hue_width": 18,                # Thang HSV của Pillow: H từ 0–255, khác OpenCV.
    "red_min_saturation": 50,
    "red_min_r_minus_g": 20,
    "red_protect_max_below": 100,       # Không xóa vùng rất tối dù màu hơi đỏ.
    "background_max_side": 768,
    "background_filter_size": 31,      # Số lẻ; áp dụng trên ảnh ước lượng nền thu nhỏ.
    "background_blur": 5.0,
    "black_point": 65,                 # Tăng: đậm hơn, có thể giữ cả chữ xuyên đậm.
    "white_point": 195,                # Giảm: trắng mạnh hơn, có thể mất nét chính nhạt.
    "tone_gamma": 1.15,
    "binary_threshold": None,          # None: Otsu; hoặc số 0–255 để thử ngưỡng cố định.
}

if not BAC_PREPROCESS_RUN:
    print("10e đang tắt; các phép tiền xử lý chưa áp dụng vào OCR hàng loạt.")
else:
    from PIL import Image
    from IPython.display import Image as NotebookImage, display

    if "bac_su_han" not in SELECTED_JOBS:
        raise ValueError("Thêm bac_su_han vào SELECTED_JOBS rồi chạy lại mục 6–7.")
    bac_samples = [i for i in selected_items if i["job"] == "bac_su_han" and i["mode"] == "ocr"]
    if (not BAC_PREPROCESS_INDICES or len(set(BAC_PREPROCESS_INDICES)) != len(BAC_PREPROCESS_INDICES)
        or any(type(i) is not int or not 0 <= i < len(bac_samples) for i in BAC_PREPROCESS_INDICES)):
        raise IndexError(f"Chọn chỉ số ảnh khác nhau từ 0 đến {len(bac_samples)-1}.")
    if BAC_PREPROCESS_ZOOM_BOX is not None and len(BAC_PREPROCESS_INDICES) != 1:
        raise ValueError("Chỉ dùng một vùng zoom khi đang chọn một ảnh.")
    for bac_index in BAC_PREPROCESS_INDICES:
        bac_sample = bac_samples[bac_index]
        bac_path = input_path(bac_sample)
        with Image.open(bac_path) as original:
            bac_variants, bac_stats = bac_su_preprocess(original, BAC_PREPROCESS_PARAMS)
        print("Nguồn:", bac_sample["relative_path"])
        print("Vùng tô hồng = mặt nạ đỏ sẽ bị thay; phải kiểm tra có chạm nét chính không.")
        bac_board = bac_su_comparison_board(bac_variants)
        buffer = io.BytesIO(); bac_board.save(buffer, format="JPEG", quality=95)
        display(NotebookImage(data=buffer.getvalue(), format="jpeg"))
        if BAC_PREPROCESS_ZOOM_BOX is not None:
            zoom = bac_su_comparison_board(bac_variants, BAC_PREPROCESS_ZOOM_BOX, (400, 600))
            buffer = io.BytesIO(); zoom.save(buffer, format="JPEG", quality=95)
            display(NotebookImage(data=buffer.getvalue(), format="jpeg"))
        print(bac_stats)
        if BAC_PREPROCESS_SAVE or BAC_PREPROCESS_OCR:
            bac_folder, bac_report = save_bac_su_trial(RUN_ROOT, bac_sample, bac_path,
                bac_variants, bac_stats, BAC_PREPROCESS_ZOOM_BOX)
            print("Đã lưu phép thử riêng:", bac_folder)
            if BAC_PREPROCESS_OCR:
                display(pd.DataFrame(compare_bac_su_trial_ocr(bac_folder, bac_report, bac_sample, bac_path)))
                print("So với ảnh gốc: số vùng/confidence/số chữ không phải độ chính xác; chưa có CER.")
        del bac_variants, bac_board


### 10f. OCR một ảnh tiền xử lý và xem như 10b

Chạy 10e với SAVE=True, nạp v6 + batch 1 ở mục 8 và helper mục 9/10b.
Chạy cell dưới với BAC_OCR_RUN=True; chọn soft_gray hoặc binary_otsu.
None dùng thư mục vừa lưu ở 10e; có thể nhập đường dẫn phép thử cũ.
Cell đọc PNG đầy đủ độ phân giải đã lưu, không tạo lại ảnh tiền xử lý.
TXT, JSON tọa độ và metadata thử lưu trong thư mục con mới, không vào bộ OCR chính.
Có thể copy nội dung cell 10f bên dưới vào cell tương ứng trên Kaggle.


In [ ]:
# ===== 10f: OCR một ảnh đã tiền xử lý, xem như 10b =====
# Copy toàn bộ file này vào một cell sau 10e. Không cần import lại notebook.
BAC_OCR_RUN = False
BAC_OCR_TRIAL_FOLDER = None       # None: thư mục vừa lưu ở 10e; hoặc đường dẫn phép thử cũ.
BAC_OCR_VARIANT = "soft_gray"     # "soft_gray", "binary_otsu", "gray_only", "original".
BAC_OCR_SHOW_CROPS = True
BAC_OCR_CROP_IDS = None           # None: vài vùng đầu theo TXT; hoặc ID trong cột index_model.
BAC_OCR_CROP_COUNT = 3
BAC_OCR_SAVE = True               # TXT/JSON/metadata nằm riêng trong thư mục phép thử.

if not BAC_OCR_RUN:
    print("10f đang tắt. Bật BAC_OCR_RUN để xem OCR một ảnh đã lưu ở 10e.")
else:
    from PIL import Image
    from IPython.display import Image as NotebookImage, display

    if globals().get("ocr") is None:
        raise RuntimeError("Nạp PaddleOCR Hán v6/batch 1 ở mục 8 và chạy helper mục 9 trước 10f.")
    if RECOGNITION_MODEL != "PP-OCRv6_medium_rec" or HAN_REC_BATCH_SIZE != 1:
        raise ValueError("10f giữ PP-OCRv6_medium_rec và batch size 1.")
    if BAC_OCR_VARIANT not in {"soft_gray", "binary_otsu", "gray_only", "original"}:
        raise ValueError("BAC_OCR_VARIANT không hợp lệ.")
    if BAC_OCR_TRIAL_FOLDER is None and "bac_folder" not in globals():
        raise RuntimeError("Chạy 10e với BAC_PREPROCESS_SAVE=True trước; hoặc nhập BAC_OCR_TRIAL_FOLDER.")
    bac_ocr_trial = Path(bac_folder if BAC_OCR_TRIAL_FOLDER is None else BAC_OCR_TRIAL_FOLDER)
    bac_ocr_report = json.loads((bac_ocr_trial / "trial.json").read_text(encoding="utf-8"))
    bac_ocr_sample = bac_ocr_report["source"]
    if bac_ocr_sample["job"] != "bac_su_han" or bac_ocr_sample["mode"] != "ocr":
        raise ValueError("10f chỉ dùng phép thử ảnh Bắc sứ Hán.")
    if BAC_OCR_TRIAL_FOLDER is None:
        if globals().get("bac_sample", {}).get("relative_path") != bac_ocr_sample["relative_path"]:
            raise RuntimeError("Thư mục 10e còn thuộc ảnh cũ. Chạy lại 10e với SAVE=True hoặc chọn thư mục rõ ràng.")
        if globals().get("BAC_PREPROCESS_PARAMS") != bac_ocr_report["preprocessing"]["params"]:
            raise RuntimeError("Tham số 10e đã đổi nhưng ảnh chưa lưu lại. Chạy lại 10e với SAVE=True.")
    bac_ocr_original = input_path(bac_ocr_sample)
    if file_hash(bac_ocr_original) != bac_ocr_report["source_sha256"]:
        raise ValueError("Ảnh nguồn đã thay đổi so với phép thử; không dùng tọa độ/kết quả cũ.")
    bac_ocr_path = (bac_ocr_original if BAC_OCR_VARIANT == "original"
                    else bac_ocr_trial / (BAC_OCR_VARIANT + ".png"))
    bac_ocr_sha = file_hash(bac_ocr_path)
    if BAC_OCR_VARIANT != "original" and bac_ocr_sha != bac_ocr_report["output_hashes"][BAC_OCR_VARIANT]["file_sha256"]:
        raise ValueError("Ảnh tiền xử lý đã bị sửa ngoài phép thử. Tạo lại ở 10e để lưu đúng tham số/hash.")
    with Image.open(bac_ocr_path) as candidate:
        if list(candidate.size) != bac_ocr_report["preprocessing"]["source_size"]:
            raise ValueError("Kích thước ảnh thử không khớp ảnh nguồn.")

    bac_ocr_text, bac_ocr_raw, bac_ocr_review = predict_image(bac_ocr_path, bac_ocr_sample, apply_bac_preprocess=False)
    bac_ocr_data = bac_ocr_raw.get("res", bac_ocr_raw)
    bac_ocr_polygons = bac_ocr_data.get("rec_polys", [])
    bac_ocr_order = bac_ocr_review["order_indices"]
    bac_ocr_rank = {index: rank+1 for rank, index in enumerate(bac_ocr_order)}
    print("Nguồn:", bac_ocr_sample["relative_path"], "| đầu vào:", BAC_OCR_VARIANT)
    print("Số xanh = dòng TXT từ 1; index_model là ID gốc trong JSON.")
    display_detection_image(bac_ocr_path, bac_ocr_polygons, order_indices=bac_ocr_order)
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame([{"dong_txt": bac_ocr_rank[index], "index_model": index,
            "chu": bac_ocr_data["rec_texts"][index], "diem_recognition": bac_ocr_data["rec_scores"][index],
            "toa_do_anh_nguon": bac_ocr_polygons[index]} for index in bac_ocr_order]))
    print("\nTXT theo thứ tự đọc:")
    print(bac_ocr_text)
    print("Cảnh báo bố cục:", bac_ocr_review["layout_flags"])
    if BAC_OCR_SHOW_CROPS:
        if type(BAC_OCR_CROP_COUNT) is not int or BAC_OCR_CROP_COUNT < 1:
            raise ValueError("BAC_OCR_CROP_COUNT phải là số nguyên dương.")
        bac_ocr_ids = bac_ocr_order[:BAC_OCR_CROP_COUNT] if BAC_OCR_CROP_IDS is None else BAC_OCR_CROP_IDS
        for row in han_crop_preview(bac_ocr_path, bac_ocr_data, bac_ocr_ids):
            index = row["index_model"]
            print("Dòng TXT", bac_ocr_rank[index], "| index_model", index,
                  "| crop W/H:", row["crop_w"], row["crop_h"])
            print(bac_ocr_data["rec_texts"][index])
            thumbnail = row["image"].copy()
            thumbnail.thumbnail((1400, 400))
            buffer = io.BytesIO()
            thumbnail.save(buffer, format="JPEG", quality=95)
            display(NotebookImage(data=buffer.getvalue(), format="jpeg"))
    if BAC_OCR_SAVE:
        bac_ocr_output = bac_ocr_trial / ("preview_ocr_" + BAC_OCR_VARIANT + "_" +
            datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
        bac_ocr_output.mkdir(parents=True, exist_ok=False)
        (bac_ocr_output / "result.txt").write_bytes(bac_ocr_text.encode("utf-8"))
        (bac_ocr_output / "result.json").write_bytes(json_bytes(bac_ocr_raw))
        (bac_ocr_output / "result.meta.json").write_bytes(json_bytes({
            "source": bac_ocr_sample, "source_sha256": bac_ocr_report["source_sha256"],
            "input_variant": BAC_OCR_VARIANT, "input_sha256": bac_ocr_sha,
            "preprocessing": bac_ocr_report["preprocessing"], "config": recognition_config(bac_ocr_sample, apply_bac_preprocess=False),
            "coordinate_space": "source_image_pixels", "review": bac_ocr_review,
            "applied_to_production": False}))
        print("Đã lưu TXT/JSON thử:", bac_ocr_output)


## 11. OCR toàn bộ phần đang chọn

Kết quả lưu sau mỗi ảnh. Chạy lại sẽ bỏ qua ảnh đã hoàn tất đúng nguồn/tham số.
Nếu một file lỗi, lưu nhật ký và tiếp tục file khác; báo cáo không coi file lỗi là đã OCR xong.
Với `MAX_IMAGES_PER_JOB` khác `None`, đây chỉ là lượt thử giới hạn.

In [ ]:
from tqdm.auto import tqdm

batch_results = []
log_path = report_dir / "nhat_ky.jsonl"
for item in tqdm(selected_items, desc="OCR / sao chép TXT"):
    row = {"job": item["job"], "file": item["relative_path"], "source_id": item["id"]}
    try:
        row.update(process_item(item))
    except Exception as exc:
        row.update(status="error", error=f"{type(exc).__name__}: {exc}")
        print("Lỗi:", row["job"], row["file"], row["error"])
    row["time_utc"] = datetime.now(timezone.utc).isoformat()
    with log_path.open("a", encoding="utf-8") as log:
        log.write(json.dumps(row, ensure_ascii=False) + "\n")
        log.flush()
    batch_results.append(row)

display(pd.DataFrame(batch_results).groupby(["job", "status"]).size().rename("so_file").reset_index())

## 12. Ghép TXT thô theo tác phẩm và phiên bản

Chỉ ghép khi **mọi ảnh trong tác phẩm thuộc kiểm kê đầy đủ** đã có kết quả hợp lệ, không phải chỉ các ảnh giới hạn ở lượt thử.
Thứ tự theo tên `phan0001`, `phan0002`…; các ảnh ngăn bằng form feed `U+000C`, không nối câu hoặc sửa đoạn.
Giữ riêng Hán/dịch âm/dịch nghĩa/dịch thơ. TXT nguồn Bắc sứ Việt đã là file riêng nên không ghép lại.

In [ ]:
def completed_bytes(item):
    paths = output_paths(item)
    if not paths["meta"].is_file():
        return None
    metadata = json.loads(paths["meta"].read_bytes().decode("utf-8"))
    config_sha = bytes_hash(json_bytes(recognition_config(item)))
    identity = metadata.get("identity", {})
    if identity.get("source_id") != item["id"] or identity.get("config_sha256") != config_sha:
        return None
    state, _ = check_saved(paths, identity)
    if state != "done":
        return None
    # Kiểm tra byte nguồn thực tế, kể cả khi đã có cache/kết quả từ lượt trước.
    if file_hash(input_path(item)) != identity.get("source_sha256"):
        return None
    return paths["txt"].read_bytes()

def merge_complete_works(items):
    groups = defaultdict(list)
    for item in items:
        if item["mode"] == "ocr":
            groups[(item["job"], item["work_stem"])].append(item)
    report = []
    for (job_key, stem), parts in groups.items():
        parts = sorted(parts, key=lambda i: natural_key(i["relative_path"]))
        try:
            contents = [completed_bytes(item) for item in parts]
        except (OSError, ValueError) as exc:
            report.append({"job": job_key, "work": stem, "status": "error", "error": str(exc)})
            continue
        if any(content is None for content in contents):
            report.append({"job": job_key, "work": stem, "status": "not_complete", "parts": len(parts)})
            continue
        target = output_paths(parts[0])["txt"].parent / f"{stem}_ocr_tho.txt"
        merged = b"\x0c".join(contents)
        if target.exists() and target.read_bytes() != merged:
            report.append({"job": job_key, "work": stem, "status": "error",
                           "error": f"File ghép đã khác nội dung, giữ nguyên: {target}. Dùng RUN_NAME mới."})
            continue
        atomic_write(target, merged)
        report.append({"job": job_key, "work": stem, "status": "merged", "parts": len(parts),
                       "file": str(target), "sha256": bytes_hash(merged),
                       "source_files": [i["relative_path"] for i in parts]})
    return report

In [ ]:
merged_works = merge_complete_works(all_items)
atomic_write(report_dir / "ghep_theo_tac_pham.json", json_bytes(merged_works))
display(pd.DataFrame(merged_works))

## 13. Báo cáo lượt chạy

Điểm tin cậy chỉ để ưu tiên xem lại, **không phải độ chính xác đã đo**. Chưa chứng nhận đủ nghĩa, dóng hàng hay quota cặp câu.
File báo lỗi/xung đột cần xử lý trước khi xem bộ OCR là đã chạy đủ. Lượt thử giới hạn không đại diện toàn bộ phần được giao.

In [ ]:
summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selected_jobs": SELECTED_JOBS, "total_inventory": len(all_items),
    "processed_this_pass": len(batch_results), "limit_per_job": MAX_IMAGES_PER_JOB,
    "statuses": dict(Counter(r["status"] for r in batch_results)),
    "empty_predictions": sum(bool(r.get("empty_prediction")) for r in batch_results),
    "needs_review_files": sum(any(r.get(k, 0) > 0 for k in ["low_confidence_count", "unknown_confidence_count", "line_warning_count"]) for r in batch_results),
    "works_merged": sum(r["status"] == "merged" for r in merged_works),
    "not_complete_works": sum(r["status"] != "merged" for r in merged_works),
    "scope": "raw OCR only; not cleaned, aligned or quota-certified",
    "versions": runtime_versions,
}
atomic_write(report_dir / "tong_ket.json", json_bytes(summary))
table = pd.DataFrame(batch_results)
atomic_write(report_dir / "ket_qua.csv", table.to_csv(index=False).encode("utf-8-sig"))
display(table)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Kết quả đã lưu:", RUN_ROOT)

## 14. Tải kết quả về máy — tùy chọn

Chạy cell này khi cần một ZIP của lượt chạy. Không chứa ảnh nguồn, trọng số model hay thông tin đăng nhập.
Colab lưu tại `OUTPUT_ROOT` trên Drive. Kaggle lưu tại `/kaggle/working`: cần Save Version hoặc tải ZIP trước khi kết thúc phiên. Không đưa output notebook có nội dung riêng tư vào Git.

In [ ]:
archive_root = Path("/kaggle/working") if RUNTIME == "kaggle" else Path("/content") if RUNTIME == "colab" else RUN_ROOT.parent
archive = shutil.make_archive(str(archive_root / RUN_NAME), "zip", root_dir=RUN_ROOT)
if RUNTIME == "colab":
    from google.colab import files
    files.download(archive)
else:
    from IPython.display import FileLink, display
    display(FileLink(archive))
    print("ZIP:", archive)
    print("Kaggle: có thể tải file trong Files/Output. Save Version để giữ output của phiên.")

## Bố cục kết quả và cách đổi dữ liệu

```text
NLP_OCR_TV02/paddle_vietocr_w512_v01/
├── CPO_09/01_nguon/viet/dich_nghia/       # TXT Việt nguồn thô, nguyên byte
├── CPO_09/02_ocr_tho/han/<mã phần>/      # OCR ảnh Hán Bắc sứ
├── CPO_11/02_ocr_tho/han/<mã bài>/
├── CPO_11/02_ocr_tho/viet/dich_am/<mã bài>/
├── CPO_11/02_ocr_tho/viet/dich_nghia/<mã bài>/
├── CPO_12/02_ocr_tho/...                 # Tương tự, thêm dịch thơ
└── 05_quan_ly/                          # Kiểm kê, nhật ký, lỗi, báo cáo
```

Mỗi ảnh có `phanXXXX_pdfpYYYY.txt`, `.json`, `.meta.json`.
File ghép theo bài/phiên bản có đuôi `_ocr_tho.txt`; các TXT theo ảnh vẫn được giữ riêng để sửa/đối chiếu.

- **Chạy tác phẩm khác:** sửa `SELECTED_JOBS`; chạy lại từ kiểm kê. Nếu đổi model/thứ tự chữ, đổi `RUN_NAME`.
- **Chạy một bài:** sửa `ONLY_WORKS`, ví dụ `{"CPO_12": ["bai0131"]}`; kiểm tra bảng trước khi OCR.
- **Đổi thư mục ảnh:** sửa `folder_id` trong JOBS, hoặc chọn `INPUT_MODE="local"` và sửa `local_folder`.
- **Chạy tiếp sau ngắt phiên:** kết nối lại Drive, nạp lại cấu hình/model, kiểm kê lại, chạy cell hàng loạt.
- **Bắc sứ Việt:** giữ `mode="text"`; không đổi thành OCR cho hai TXT có sẵn.
- **Chữ Nôm/chữ cổ và cột dọc:** phải xem ảnh thử; hỗ trợ Hán phồn thể không có nghĩa mọi chữ Nôm đều nhận đúng.

### Tài liệu chính thức

- [PaddleOCR và mã nguồn](https://github.com/PaddlePaddle/PaddleOCR)
- [API OCR và cấu trúc kết quả](https://www.paddleocr.ai/latest/en/version3.x/pipeline_usage/OCR.html)
- [Engine Transformers](https://www.paddleocr.ai/latest/en/version3.x/inference_deployment/local_inference/inference_engine.html)
- [Drive API: phân trang danh sách file](https://developers.google.com/workspace/drive/api/reference/rest/v3/files/list)
- [Colab: Drive, cache cục bộ và giới hạn runtime](https://research.google.com/colaboratory/faq.html)

Notebook đã được kiểm tra cú pháp và luồng xử lý với dữ liệu mô phỏng ở máy phát triển.
Chưa chạy PaddleOCR thực tế trên GPU Kaggle/Colab; chất lượng nhận dạng cần đánh giá trên ảnh thử của bạn.

Mỗi ảnh OCR: `.txt` chứa chữ; `.json` chứa vùng `rec_polys` theo ảnh nguồn và kết quả model;
`.meta.json` chứa `review.reading_map` với `txt_line` (từ 1), `index_model` (từ 0) và `polygon`.
Lý–Trần bổ sung `line_details.crop_box`, hash pixel crop, `row_detection` và các kết quả recognition gốc.
`layout_flags`/`line_warning_count` là gợi ý rà soát, không phải độ chính xác OCR.
